<a href="https://colab.research.google.com/github/DanisStepanov/image_processing/blob/main/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_%E2%84%966_%D0%A3%D1%82%D0%BE%D1%87%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5_%D1%80%D0%B5%D0%BB%D1%8C%D0%B5%D1%84%D0%B0_%D0%B8_%D0%B0%D0%BD%D0%BE%D0%BC%D0%B0%D0%BB%D0%B8%D0%B8_%D0%91%D1%83%D0%B3%D0%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Практическая работа №6. Уточнение цифровой модели рельефа по спутниковым снимкам и уточнение аномалий Буге для выбранной территории**

## **1. Цель работы**

Построить для выбранной территории полную цепочку обработки спутниковых изображений — от безоблачного композита и производных рельефа до уточнённой модели рельефа и карты аномалий Буге — и по полученным числам и изображениям оценить, как ошибка глобальной модели рельефа в лесу влияет на прикладные расчёты и гравиметрические аномалии.

## **2. Задачи работы**

1.  **Выбрать** зону из списка, рассмотреть её на подложке Esri и записать гипотезы о том, где глобальная модель рельефа ошибается сильнее всего;
2.  **Получить** безоблачный композит Sentinel-2, самостоятельно рассчитать спектральные индексы, производные рельефа и фильтры низких и высоких частот, построить трёхмерные поверхности;
3.  **Самостоятельно отфильтровать и очистить** лазерные измерения ICESat-2: выбрать порог качества, записать маски правил очистки, оценить завышение модели рельефа по классам покрова;
4.  **Самостоятельно обучить и сравнить** все классификаторы из лекции по GEE (Random Forest, CART, градиентный бустинг, SVM, наивный Байес, минимальное расстояние) с пространственной проверкой, построить матрицу ошибок, рассчитать метрики по классам и карту покрова лучшим классификатором;
5.  **Самостоятельно написать** показатели точности высот, обучить и сравнить четыре модели регрессии для уточнения рельефа и построить уточнённый рельеф лучшей из них;
6.  **Самостоятельно реализовать** расчёт аномалий Буге (сглаживание, вычитание притяжения слоя, гауссов фон, пороговая сегментация) и проверить чувствительность к параметрам;
7.  **Решить** прикладную задачу своей зоны;
8.  **Интерпретировать** результаты: ответить на вопросы под выводами ячеек и сформулировать итоговые выводы с конкретными числами.

> **Примечание.** Работа выполняется по материалам:
> - лекция «Цифровая обработка спутниковых изображений рельефа и гравитационного поля» — **все разделы**; готовые ячейки ниже взяты из неё, нужные разделы указаны в подсказках;
> - [лекция по GEE и машинному обучению](https://colab.research.google.com/drive/1BU-Q4D0XEIy2KiEtzw-3dKISE0lwGLV2) и [лекция по кластеризации](https://colab.research.google.com/drive/1rWGeyE30g6rAk6eYgdLkFdEFaiVZ3cey) — подключение к GEE, композиты, классификация, K-средних;
> - [практическая работа №5](https://colab.research.google.com/drive/1mwCV-82k9R5ZtR0NGVJT3oLNyKDG0ZXO) — случайная и пространственная проверка точности.

> 📝 **Как выполнять работу.** Выполняйте ячейки по порядку. Готовые ячейки просто запускайте. В ячейках с пометкой `# Ваш код` и с `...` в заготовке код пишете вы; имена переменных, указанные в задании (например, `isat_q`, `cls`, `best_reg`, `r1`), сохраняйте — их используют следующие готовые ячейки. Под каждым важным результатом стоит вопрос: ответ пишите прямо в блокноте, в ячейку **Ответ:** (двойной щелчок по ячейке, текст ответа, затем Shift+Enter). Отвечайте по тому, что вывелось у вас: опишите карту или график, приведите числа из таблиц, объясните причину. Сдаётся только выполненный блокнот.

---

## **3. Выбор территории и подготовка**

### **3.1. Настройка рабочей среды и аутентификация**

In [ ]:
%pip install -q geemap sliderule==5.6.3 pyproj rasterio folium plotly geopandas

In [ ]:
# Импорт библиотек
import os, re, math, time, warnings, urllib.request                 # Файлы, текст, время, загрузка по ссылке.
import numpy as np                                                       # Массивы и расчёты на сетке.
import pandas as pd                                                      # Таблицы.
import matplotlib.pyplot as plt                                          # Графики и карты.
from matplotlib.colors import ListedColormap, BoundaryNorm, LightSource, to_rgb   # Палитры, отмывка рельефа, цвета.
from matplotlib.patches import Patch                                     # Элементы легенд.
import ee, geemap, folium, pyproj                                        # Earth Engine, интерактивные карты, системы координат.
import plotly.graph_objects as go                                        # Трёхмерная модель.
import plotly.io as pio                                                  # Настройка вывода графиков plotly.
from pyproj import Transformer                                           # Пересчёт координат и высот.
from scipy.ndimage import uniform_filter, gaussian_filter, map_coordinates, label   # Сглаживание, интерполяция, связные области.
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingRegressor, HistGradientBoostingClassifier  # Случайный лес и градиентный бустинг.
from sklearn.linear_model import LogisticRegression                      # Логистическая регрессия.
from sklearn.svm import SVC                                              # Метод опорных векторов.
from sklearn.pipeline import make_pipeline                               # Цепочка «стандартизация → модель».
from sklearn.preprocessing import StandardScaler                         # Стандартизация признаков.
from sklearn.model_selection import GroupKFold, KFold                    # Перекрёстная проверка: по пространственным блокам и случайная.
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, precision_score, recall_score  # Метрики классификации.
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay     # Матрица ошибок классификации.
from sklearn.cluster import KMeans                                       # Кластеризация k-средних.
from sklearn.metrics import silhouette_score                             # Коэффициент силуэта: контроль качества кластеризации.
from sklearn.inspection import permutation_importance                   # Важность признаков перестановкой.

warnings.filterwarnings("ignore")                                       # Не выводим служебные предупреждения библиотек.
pio.renderers.default = "colab+plotly_mimetype"                         # Графики plotly видны и в Colab, и в VS Code/Jupyter.
pyproj.network.set_network_enabled(True)                                 # Сетки геоида загружаются автоматически.
SEED = 42                                                                # Начальное значение генератора случайных чисел: результаты воспроизводимы.
N_JOBS = int(os.environ.get("NB_N_JOBS", "-1"))                          # Число потоков для обучения моделей (−1 – все).
OUT = "data"                                                             # Папка для загруженных данных (лазерные измерения, фотоны).
os.makedirs(OUT, exist_ok=True)                                          # Создаём папку, если её нет.
print("ee", ee.__version__, "| geemap", geemap.__version__, "| pyproj", pyproj.__version__)  # Версии ключевых библиотек.
%config InlineBackend.figure_format = "jpeg"                             # Рисунки со снимками компактнее в JPEG.

Перед началом работы с geemap и Google Earth Engine необходимо:

- **Зарегистрировать аккаунт Google Earth Engine**: Перейдите на [https://code.earthengine.google.com/register](https://code.earthengine.google.com/register) для создания аккаунта GEE. После регистрации нужно настроить проект Google Cloud и активировать API Earth Engine, следуя инструкциям [здесь](https://docs.google.com/document/d/1ZGSmrNm6_baqd8CHt33kIBWOlvkh-HLr46bODgJN1h0/edit?usp=sharing). GEE бесплатен для [некоммерческого и исследовательского использования](https://earthengine.google.com/noncommercial).

- **Проверить аутентификацию API**: Чтобы убедиться в правильности настройки аккаунта Earth Engine, запустите [этот тестовый блокнот](https://colab.research.google.com/github/gee-community/geemap/blob/master/docs/notebooks/geemap_colab.ipynb) в Google Colab.

In [ ]:
PROJECT_ID = 'cedar-bot-510810-r8'
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print("Earth Engine подключён:", ee.Number(1).add(1).getInfo() == 2)

### **3.2. Выбор зоны**

Выберите одну зону из списка (номер согласуйте с преподавателем, чтобы зоны в группе не повторялись). Для всех 70 зон лазерные измерения ICESat-2 и фрагмент гравитационного поля GGMplus подготовлены заранее и загружаются из репозитория за секунды; для этого границы `ROI_BOUNDS` нужно скопировать **точно**, без изменений. У каждой зоны своя прикладная задача (раздел 10) и свои параметры к ней.

<details>
<summary><b>Список зон</b> (нажмите, чтобы развернуть)</summary>

| № | Территория | Прикладная задача | `ROI_BOUNDS` | Параметры прикладной задачи |
|---|---|---|---|---|
| 1 | Пойма Оби у Барнаула | Оценка зон затопления | `[83.55, 53.25, 83.95, 53.45]` | `WATER_RISE_M = 5.0`; `WATER_POINT = (83.821, 53.3494)` |
| 2 | Слияние Тобола и Иртыша у Тобольска | Оценка зон затопления | `[68.05, 58.10, 68.45, 58.30]` | `WATER_RISE_M = 8.0`; `WATER_POINT = (68.2142, 58.1857)` |
| 3 | Пойма Амура у Хабаровска | Оценка зон затопления | `[134.85, 48.38, 135.20, 48.58]` | `WATER_RISE_M = 5.0`; `WATER_POINT = (135.0385, 48.4681)` |
| 4 | Пойма Оки у Рязани | Оценка зон затопления | `[39.65, 54.55, 40.05, 54.75]` | `WATER_RISE_M = 6.0`; `WATER_POINT = (39.8035, 54.6552)` |
| 5 | Слияние Бии и Катуни у Бийска | Оценка зон затопления | `[84.95, 52.40, 85.35, 52.60]` | `WATER_RISE_M = 6.0`; `WATER_POINT = (85.1783, 52.5079)` |
| 6 | Долины Белой и Уфы у Уфы | Оценка зон затопления | `[55.80, 54.65, 56.20, 54.85]` | `WATER_RISE_M = 8.0`; `WATER_POINT = (55.8936, 54.725)` |
| 7 | Трасса М-5 через хребты Южного Урала (Аша – Сим) | Профиль линейного объекта (дорога, ЛЭП, трубопровод) | `[57.20, 54.90, 57.70, 55.10]` | `PROFILE = [(57.27, 55.0), (57.66, 54.99)]` |
| 8 | Байкало-Амурская магистраль у Тынды | Профиль линейного объекта (дорога, ЛЭП, трубопровод) | `[124.50, 55.05, 124.90, 55.25]` | `PROFILE = [(124.53, 55.1), (124.87, 55.18)]` |
| 9 | Долина Мзымты у Красной Поляны | Профиль линейного объекта (дорога, ЛЭП, трубопровод) | `[40.00, 43.55, 40.35, 43.75]` | `PROFILE = [(40.04, 43.58), (40.3, 43.68)]` |
| 10 | Горная Шория: долина Кондомы у Мундыбаша | Профиль линейного объекта (дорога, ЛЭП, трубопровод) | `[87.15, 53.10, 87.55, 53.30]` | `PROFILE = [(87.18, 53.27), (87.52, 53.13)]` |
| 11 | Чуйский тракт у Семинского перевала | Профиль линейного объекта (дорога, ЛЭП, трубопровод) | `[85.45, 50.93, 85.80, 51.13]` | `PROFILE = [(85.67, 51.12), (85.56, 50.95)]` |
| 12 | Яблоновый хребет западнее Читы | Профиль линейного объекта (дорога, ЛЭП, трубопровод) | `[113.05, 51.95, 113.45, 52.15]` | `PROFILE = [(113.08, 52.05), (113.42, 52.04)]` |
| 13 | Коркинский угольный разрез | Объёмы земляных масс | `[61.20, 54.78, 61.60, 54.98]` | `VOLUME_BOX = [61.3943, 54.8952, 61.4502, 54.9238]`; `VOLUME_LEVEL = 236.0` |
| 14 | Бачатский угольный разрез (Кузбасс) | Объёмы земляных масс | `[86.00, 54.18, 86.40, 54.38]` | `VOLUME_BOX = [86.138, 54.2537, 86.1973, 54.2945]`; `VOLUME_LEVEL = 290.0` |
| 15 | Сибайский медноколчеданный карьер | Объёмы земляных масс | `[58.48, 52.62, 58.88, 52.82]` | `VOLUME_BOX = [58.6261, 52.6867, 58.6523, 52.7092]`; `VOLUME_LEVEL = 380.0` |
| 16 | Асбестовский карьер (Баженовское месторождение) | Объёмы земляных масс | `[61.25, 56.93, 61.65, 57.13]` | `VOLUME_BOX = [61.4745, 56.978, 61.5182, 57.0457]`; `VOLUME_LEVEL = 233.0` |
| 17 | Бородинский угольный разрез (Канско-Ачинский бассейн) | Объёмы земляных масс | `[94.65, 55.80, 95.05, 56.00]` | `VOLUME_BOX = [94.8672, 55.833, 94.9753, 55.8662]`; `VOLUME_LEVEL = 338.0` |
| 18 | Коршуновский железорудный карьер (Иркутская область) | Объёмы земляных масс | `[103.95, 56.48, 104.35, 56.68]` | `VOLUME_BOX = [104.095, 56.5281, 104.1513, 56.5593]`; `VOLUME_LEVEL = 484.0` |
| 19 | Нижний Тагил и Высокогорское железорудное месторождение | Углублённый гравиметрический анализ | `[59.80, 57.82, 60.20, 58.02]` | — |
| 20 | Лакколиты Кавказских Минеральных Вод | Углублённый гравиметрический анализ | `[42.85, 43.95, 43.25, 44.15]` | — |
| 21 | Тункинская рифтовая впадина | Углублённый гравиметрический анализ | `[102.15, 51.62, 102.55, 51.82]` | — |
| 22 | Ильменский хребет у Миасса | Углублённый гравиметрический анализ | `[59.95, 54.95, 60.35, 55.15]` | — |
| 23 | Алданский щит у Алдана | Углублённый гравиметрический анализ | `[125.20, 58.50, 125.60, 58.70]` | — |
| 24 | Жигулёвские горы | Углублённый гравиметрический анализ | `[49.45, 53.30, 49.85, 53.50]` | — |
| 25 | Центрально-Лесной заповедник | Высота растительности | `[32.75, 56.40, 33.15, 56.60]` | — |
| 26 | Висимский заповедник (Средний Урал) | Высота растительности | `[59.40, 57.32, 59.80, 57.52]` | — |
| 27 | Сихотэ-Алинский заповедник | Высота растительности | `[135.90, 45.10, 136.30, 45.30]` | — |
| 28 | Бузулукский бор | Высота растительности | `[52.05, 52.75, 52.45, 52.95]` | — |
| 29 | Верховья Белой у Гузерипля (Кавказский заповедник) | Высота растительности | `[39.95, 43.90, 40.35, 44.10]` | — |
| 30 | Неруссо-Деснянское полесье (Брянский лес) | Высота растительности | `[33.80, 52.38, 34.20, 52.58]` | — |
| 31 | Пойма Оби у Колпашево | Оценка зон затопления | `[82.70, 58.20, 83.10, 58.40]` | `WATER_RISE_M = 4.0`; `WATER_POINT = (82.9773, 58.31)` |
| 32 | Пойма Иртыша у Тары | Оценка зон затопления | `[74.15, 56.80, 74.55, 57.00]` | `WATER_RISE_M = 6.0`; `WATER_POINT = (74.427, 56.9092)` |
| 33 | Пойма Вятки у Кирова | Оценка зон затопления | `[49.45, 58.50, 49.85, 58.70]` | `WATER_RISE_M = 5.0`; `WATER_POINT = (49.6819, 58.6295)` |
| 34 | Протоки Енисея у Минусинска | Оценка зон затопления | `[91.50, 53.60, 91.90, 53.80]` | `WATER_RISE_M = 6.0`; `WATER_POINT = (91.6489, 53.7159)` |
| 35 | Пойма Зеи у Свободного | Оценка зон затопления | `[127.95, 51.30, 128.35, 51.50]` | `WATER_RISE_M = 4.0`; `WATER_POINT = (128.1767, 51.3721)` |
| 36 | Пойма Хопра у Новохопёрска (Хопёрский заповедник) | Оценка зон затопления | `[41.45, 51.00, 41.85, 51.20]` | `WATER_RISE_M = 3.0`; `WATER_POINT = (41.6226, 51.1133)` |
| 37 | Пойма Чулыма у Асино | Оценка зон затопления | `[85.95, 56.90, 86.35, 57.10]` | `WATER_RISE_M = 5.0`; `WATER_POINT = (86.1544, 57.0669)` |
| 38 | Долина Селенги у Улан-Удэ | Оценка зон затопления | `[107.40, 51.75, 107.80, 51.95]` | `WATER_RISE_M = 3.0`; `WATER_POINT = (107.5609, 51.8237)` |
| 39 | Трасса Р-258 «Байкал» у Култука и Слюдянки | Профиль линейного объекта (дорога, ЛЭП, трубопровод) | `[103.50, 51.62, 103.90, 51.82]` | `PROFILE = [(103.53, 51.7), (103.87, 51.78)]` |
| 40 | Трасса М-5 через хребты у Златоуста | Профиль линейного объекта (дорога, ЛЭП, трубопровод) | `[59.45, 55.05, 59.90, 55.25]` | `PROFILE = [(59.48, 55.13), (59.87, 55.13)]` |
| 41 | Трасса Туапсе – Хадыженск через Главный Кавказский хребет | Профиль линейного объекта (дорога, ЛЭП, трубопровод) | `[39.05, 44.10, 39.45, 44.30]` | `PROFILE = [(39.1, 44.13), (39.4, 44.28)]` |
| 42 | Трасса Р-297 «Амур» через Малый Хинган у Облучья | Профиль линейного объекта (дорога, ЛЭП, трубопровод) | `[130.85, 48.90, 131.25, 49.10]` | `PROFILE = [(130.88, 49.05), (131.22, 48.95)]` |
| 43 | Трасса Р-257 «Енисей» через Западный Саян у Танзыбея | Профиль линейного объекта (дорога, ЛЭП, трубопровод) | `[92.80, 52.85, 93.20, 53.05]` | `PROFILE = [(92.95, 53.03), (93.1, 52.87)]` |
| 44 | Трасса Р-255 «Сибирь» через Кемчугскую возвышенность | Профиль линейного объекта (дорога, ЛЭП, трубопровод) | `[91.35, 56.05, 91.85, 56.25]` | `PROFILE = [(91.4, 56.18), (91.8, 56.12)]` |
| 45 | Трасса Р-255 «Сибирь» у Нижнеудинска | Профиль линейного объекта (дорога, ЛЭП, трубопровод) | `[98.60, 54.80, 99.10, 55.00]` | `PROFILE = [(98.65, 54.95), (99.05, 54.85)]` |
| 46 | Трасса Р-297 «Амур» и Транссиб у Могочи | Профиль линейного объекта (дорога, ЛЭП, трубопровод) | `[119.50, 53.62, 120.00, 53.82]` | `PROFILE = [(119.55, 53.77), (119.95, 53.7)]` |
| 47 | Карьер Лебединского ГОКа (Губкин) | Объёмы земляных масс | `[37.42, 51.11, 37.88, 51.35]` | `VOLUME_BOX = [37.5616, 51.2559, 37.6801, 51.3187]`; `VOLUME_LEVEL = 199.0` |
| 48 | Карьер Михайловского ГОКа (Железногорск) | Объёмы земляных масс | `[35.15, 52.27, 35.55, 52.47]` | `VOLUME_BOX = [35.402, 52.2944, 35.4529, 52.3459]`; `VOLUME_LEVEL = 216.0` |
| 49 | Гусевогорские карьеры Качканарского ГОКа | Объёмы земляных масс | `[59.35, 58.62, 59.75, 58.82]` | `VOLUME_BOX = [59.4615, 58.7122, 59.5803, 58.7906]`; `VOLUME_LEVEL = 305.0` |
| 50 | Учалинский медноколчеданный карьер | Объёмы земляных масс | `[59.20, 54.22, 59.60, 54.42]` | `VOLUME_BOX = [59.4118, 54.2989, 59.435, 54.3201]`; `VOLUME_LEVEL = 557.0` |
| 51 | Угольные разрезы у Киселёвска (Кузбасс) | Объёмы земляных масс | `[86.42, 53.93, 86.82, 54.13]` | `VOLUME_BOX = [86.4955, 54.0335, 86.5886, 54.0838]`; `VOLUME_LEVEL = 350.0` |
| 52 | Нерюнгринский угольный разрез | Объёмы земляных масс | `[124.41, 56.61, 124.81, 56.81]` | `VOLUME_BOX = [124.526, 56.6784, 124.6647, 56.7349]`; `VOLUME_LEVEL = 839.0` |
| 53 | Тугнуйский угольный разрез (Бурятия) | Объёмы земляных масс | `[108.31, 51.21, 108.71, 51.41]` | `VOLUME_BOX = [108.4553, 51.2944, 108.5554, 51.3317]`; `VOLUME_LEVEL = 867.0` |
| 54 | Железорудные карьеры у Бакала (Южный Урал) | Объёмы земляных масс | `[58.60, 54.85, 59.00, 55.05]` | `VOLUME_BOX = [58.7284, 54.9038, 58.8401, 54.9602]`; `VOLUME_LEVEL = 648.0` |
| 55 | Кытлымский массив Платиноносного пояса Урала | Углублённый гравиметрический анализ | `[59.27, 59.35, 59.77, 59.60]` | — |
| 56 | Баргузинская впадина | Углублённый гравиметрический анализ | `[109.95, 53.55, 110.35, 53.75]` | — |
| 57 | Вишнёвогорский щелочной массив | Углублённый гравиметрический анализ | `[60.45, 55.90, 60.85, 56.10]` | — |
| 58 | Чинейский расслоенный массив (Забайкалье) | Углублённый гравиметрический анализ | `[118.20, 56.40, 118.60, 56.60]` | — |
| 59 | Хребет Хамар-Дабан у Байкальска | Углублённый гравиметрический анализ | `[104.30, 51.30, 104.70, 51.50]` | — |
| 60 | Дальнегорский рудный район (Сихотэ-Алинь) | Углублённый гравиметрический анализ | `[135.35, 44.45, 135.75, 44.65]` | — |
| 61 | Гора Магнитная и Магнитогорск | Углублённый гравиметрический анализ | `[58.80, 53.33, 59.20, 53.53]` | — |
| 62 | Хребет Ямантау (Южно-Уральский заповедник) | Углублённый гравиметрический анализ | `[57.95, 54.15, 58.35, 54.35]` | — |
| 63 | Приокско-Террасный заповедник | Высота растительности | `[37.40, 54.80, 37.80, 55.00]` | — |
| 64 | Мордовский заповедник | Высота растительности | `[43.05, 54.65, 43.45, 54.85]` | — |
| 65 | Керженский заповедник | Высота растительности | `[44.40, 56.40, 44.80, 56.60]` | — |
| 66 | Усманский бор (Воронежский заповедник) | Высота растительности | `[39.40, 51.80, 39.80, 52.00]` | — |
| 67 | Башкирский заповедник | Высота растительности | `[57.30, 52.95, 57.70, 53.15]` | — |
| 68 | Кедровая Падь (национальный парк «Земля леопарда») | Высота растительности | `[131.30, 43.05, 131.70, 43.25]` | — |
| 69 | Заповедник Кузнецкий Алатау | Высота растительности | `[88.40, 54.40, 88.80, 54.60]` | — |
| 70 | Заповедник Басеги (Средний Урал) | Высота растительности | `[58.35, 58.75, 58.75, 58.95]` | — |

</details>

**Задание 3.2.** Впишите в ячейку название территории `TERRITORY` и границы `ROI_BOUNDS` своей зоны, а также параметры её прикладной задачи (если они указаны в таблице). Остальные строки не меняйте.

In [ ]:
TERRITORY = "..."                                  # Название вашей зоны.
ROI_BOUNDS = [...]                                 # Границы зоны из таблицы: запад, юг, восток, север.
S2_PERIOD = ("2023-06-15", "2023-08-31")           # Летний период для композита Sentinel-2.
ICESAT_PERIOD = ("2018-10-14", "2025-12-31")       # Период лазерных измерений ICESat-2.
WATER_RISE_M = 3.0                                 # Затопление: подъём уровня воды, м.
WATER_POINT = None                                 # Затопление: точка на водном объекте (долгота, широта) или None.
PROFILE = None                                     # Профиль: две точки [(долгота, широта), (долгота, широта)] или None.
VOLUME_BOX = None                                  # Объёмы: границы участка (запад, юг, восток, север) или None.
VOLUME_LEVEL = None                                # Объёмы: отметка выемки, м, или None.

In [ ]:
REGION = ee.Geometry.Rectangle(ROI_BOUNDS)                               # Территория как геометрия Earth Engine.
center = [(ROI_BOUNDS[1] + ROI_BOUNDS[3]) / 2, (ROI_BOUNDS[0] + ROI_BOUNDS[2]) / 2]   # Центр участка: широта, долгота.
m = geemap.Map(center=center, zoom=11)                                   # Интерактивная карта geemap.
m.add_basemap("Esri.WorldImagery")                                       # Подложка – космические снимки Esri.
m.add_layer(REGION, {"color": "yellow"}, "Участок")                      # Границы участка.
m                                                                        # Показываем карту.

**Вопрос:** Опишите территорию по подложке Esri: где лес, вода, застройка, открытые участки, каков рельеф (по виду снимка и теням)? Где, по-вашему, глобальная модель рельефа может завышать высоты сильнее всего и почему? Какая прикладная задача у вашей зоны и какой объект для неё здесь есть?

**Ответ:** _______________

### **3.3. Готовые функции**

> 📚 **Подсказка:** лекция «Цифровая обработка спутниковых изображений рельефа и гравитационного поля», **раздел 1.4** — что делает каждая функция.

In [ ]:
# Готовые функции: сетка участка и загрузка из Google Earth Engine
def utm_epsg(lon, lat):
    """Код EPSG зоны UTM для точки."""
    zone = int((lon + 180) // 6) + 1                                     # Номер шестиградусной зоны UTM.
    return f"EPSG:{(32600 if lat >= 0 else 32700) + zone}"               # 326xx – северное полушарие, 327xx – южное.

def make_grid(bounds, res=30):
    """Общая сетка участка в проекции UTM с шагом res метров."""
    lon0, lat0, lon1, lat1 = bounds                                      # Запад, юг, восток, север.
    crs = utm_epsg((lon0 + lon1) / 2, (lat0 + lat1) / 2)                 # Зона UTM по центру участка.
    to_utm = Transformer.from_crs("EPSG:4326", crs, always_xy=True)      # Пересчёт градусов в метры UTM.
    xs, ys = to_utm.transform([lon0, lon1, lon0, lon1], [lat0, lat0, lat1, lat1])   # Четыре угла участка.
    x0, x1 = math.floor(min(xs) / res) * res, math.ceil(max(xs) / res) * res        # Границы, кратные шагу сетки.
    y0, y1 = math.floor(min(ys) / res) * res, math.ceil(max(ys) / res) * res  # То же по вертикали.
    W, H = int((x1 - x0) / res), int((y1 - y0) / res)                    # Число столбцов и строк.
    return dict(crs=crs, x0=x0, y1=y1, res=res, W=W, H=H)               # Левый верхний угол, шаг и размер сетки.

def gee_to_numpy(image, grid, tile=400):
    """Загружает многоканальное изображение GEE на сетку участка; возвращает словарь {канал: массив}."""
    bands = image.bandNames().getInfo()                                  # Имена каналов изображения.
    out = np.full((len(bands), grid["H"], grid["W"]), np.nan, np.float32)   # Пустой массив под все каналы.
    for r0 in range(0, grid["H"], tile):                                 # Обходим участок фрагментами по 400 строк
        for c0 in range(0, grid["W"], tile):                             # и 400 столбцов.
            h, w = min(tile, grid["H"] - r0), min(tile, grid["W"] - c0)  # У края фрагмент меньше.
            transform = {"scaleX": grid["res"], "shearX": 0, "translateX": grid["x0"] + c0 * grid["res"],   # Привязка фрагмента:
                         "shearY": 0, "scaleY": -grid["res"], "translateY": grid["y1"] - r0 * grid["res"]}  # угол и шаг в метрах.
            request = {"expression": image.toFloat(), "fileFormat": "NUMPY_NDARRAY",   # Что считать и в каком виде вернуть.
                       "grid": {"dimensions": {"width": w, "height": h}, "affineTransform": transform, "crsCode": grid["crs"]}}
            arr = ee.data.computePixels(request)                         # Значения пикселей фрагмента.
            for k, b in enumerate(bands):                                # Для каждого канала:
                out[k, r0:r0 + h, c0:c0 + w] = arr[b]                    # Раскладываем по каналам.
    return dict(zip(bands, out))                                         # Словарь {имя канала: массив}.

def s2_composite(region, start, end, cs_threshold=0.6):
    """Безоблачный медианный композит Sentinel-2 с индексами; маска облаков по Cloud Score+."""
    def mask_clouds(img):
        return img.updateMask(img.select("cs_cdf").gte(cs_threshold))   # Оставляем пиксели с оценкой ясности ≥ порога.
    cs = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")  # Оценки ясности неба для каждого снимка.
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")             # Коллекция снимков Sentinel-2 L2A:
           .filterBounds(region).filterDate(start, end)                  # Снимки над участком за период.
           .linkCollection(cs, ["cs_cdf"])                               # Добавляем к каждому снимку канал ясности cs_cdf.
           .map(mask_clouds))                                            # Маскируем облака и тени.
    img = col.select(["B2", "B3", "B4", "B8", "B11", "B12"]).median().divide(10000)   # Медиана по снимкам, отражение 0–1.
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")         # Вегетационный индекс.
    ndwi = img.normalizedDifference(["B3", "B8"]).rename("NDWI")         # Водный индекс.
    ndbi = img.normalizedDifference(["B11", "B8"]).rename("NDBI")        # Индекс застройки.
    n_clear = col.select("B4").count().rename("n_clear")                 # Число безоблачных наблюдений в пикселе.
    return img.addBands([ndvi, ndwi, ndbi, n_clear]), col.size().getInfo()   # Композит и число снимков.

def collection_mosaic(collection_id, band, region, name):
    """Мозаика тайлов модели рельефа в её исходной проекции с билинейной интерполяцией."""
    col = ee.ImageCollection(collection_id).filterBounds(region).select(band)   # Тайлы над участком.
    mosaic = col.mosaic().setDefaultProjection(col.first().projection())         # Склеиваем, сохраняя исходную сетку.
    return mosaic.resample("bilinear").rename(name)                              # Билинейная интерполяция при пересчёте.

def dem_images(region):
    """Четыре глобальные модели рельефа с билинейной интерполяцией."""
    cop = collection_mosaic("COPERNICUS/DEM/GLO30_2024_1", "DEM", region, "cop")        # Copernicus GLO-30.
    alos = collection_mosaic("JAXA/ALOS/AW3D30/V4_1", "DSM", region, "alos")             # ALOS AW3D30.
    merit = ee.Image("MERIT/DEM/v1_0_3").select("dem").resample("bilinear").rename("merit")              # MERIT DEM.
    nasadem = ee.Image("NASA/NASADEM_HGT/001").select("elevation").resample("bilinear").rename("nasadem")   # NASADEM.
    return cop.addBands(alos).addBands(merit).addBands(nasadem)          # Одно изображение с четырьмя каналами.

def auxiliary_images():
    """Карта покрова WorldCover, высота леса ETH, год потери леса Hansen."""
    wc = ee.ImageCollection("ESA/WorldCover/v200").first().rename("wc")  # Классы покрова, 10 м.
    wc = wc.reduceResolution(ee.Reducer.mode(), maxPixels=64)            # На сетку 30 м – самый частый класс.
    eth = ee.Image("users/nlang/ETH_GlobalCanopyHeight_2020_10m_v1").rename("eth").unmask(0)   # Высота леса, м (вне леса 0).
    eth = eth.reduceResolution(ee.Reducer.mean(), maxPixels=64)          # На сетку 30 м – средняя высота.
    loss = ee.Image("UMD/hansen/global_forest_change_2025_v1_13").select("lossyear").unmask(0).rename("lossyear")   # Год потери леса.
    return wc.addBands(eth).addBands(loss)                               # Три слоя одним изображением.

In [ ]:
# Готовые функции: лазерные высоты ICESat-2, системы высот, гравитационное поле GGMplus
DATA_URL = "https://raw.githubusercontent.com/tester170/geoai-internship-data/main"   # Подготовленные данные для Красноярска и 70 зон практической работы.

def roi_key(bounds):
    """Имя файла подготовленных данных: границы участка «запад_юг_восток_север»."""
    return "_".join(f"{v:.2f}" for v in bounds)

NOT_PREPARED = set()                                                     # Адреса, по которым подготовленных данных нет.

def fetch_prepared(folder, ext, path):
    """Скачивает подготовленный файл участка ROI_BOUNDS в path; False, если для участка его нет (другой участок)."""
    url = f"{DATA_URL}/{folder}/{roi_key(ROI_BOUNDS)}.{ext}"     # Адрес файла в репозитории.
    if url in NOT_PREPARED:                                              # Уже проверено: файла нет,
        return False                                                     # повторно не запрашиваем.
    try:
        urllib.request.urlretrieve(url, path + ".part")                  # Скачиваем во временный файл
        os.replace(path + ".part", path)                                 # и переименовываем целиком.
        return True
    except Exception:                                                    # Файла нет (другой участок) или нет связи:
        NOT_PREPARED.add(url)                                            # данные будут получены напрямую из сервиса.
        return False

def files_ru(n):
    """«1 файл», «3 файла», «5 файлов»."""
    word = "файл" if n % 10 == 1 and n % 100 != 11 else "файла" if n % 10 in (2, 3, 4) and n % 100 not in (12, 13, 14) else "файлов"
    return f"{n} {word}"

def ggm_source():
    """Откуда получено поле GGMplus: подготовленный фрагмент (участок из списка зон) или тайлы сервера Curtin University."""
    prep = os.path.join("ggmplus_cache", f"{roi_key(ROI_BOUNDS)}.npz")  # Подготовленный фрагмент поля для участка.
    tiles = sorted(f for f in os.listdir("ggmplus_cache") if f.endswith((".dg", ".ha"))) if os.path.isdir("ggmplus_cache") else []
    return ("подготовленные данные", prep) if os.path.exists(prep) and not tiles else ("тайлы", tiles)

def load_icesat2(bounds, t0, t1, cache=None):
    """Сегменты ICESat-2 ATL08 (100 м) по границам участка через сервис SlideRule. Повторный вызов читает кэш."""
    if cache and os.path.exists(cache):                                  # Данные уже скачаны:
        return pd.read_parquet(cache)                                    # читаем файл, сервис не вызываем.
    if cache and list(bounds) == list(ROI_BOUNDS) and fetch_prepared("icesat2", "parquet", cache):   # Участок из списка зон:
        if "2018-10-14" <= t0 and t1 <= "2025-12-31":                    # Период внутри подготовленного (14.10.2018–31.12.2025):
            df = pd.read_parquet(cache)                                  # тот же запрос, выполненный заранее,
            df = df[(df["time"] >= t0) & (df["time"] < pd.Timestamp(t1) + pd.Timedelta(days=1))]   # за выбранный период.
            df.to_parquet(cache)
            print("Сегменты ICESat-2 взяты из подготовленных данных")
            return df
        os.remove(cache)                                                 # Период шире подготовленного: запрос к сервису.
    from sliderule import sliderule, icesat2                             # Клиент сервиса SlideRule.
    sliderule.init("slideruleearth.io", verbose=False)                   # Подключение к открытому серверу.
    lon0, lat0, lon1, lat1 = bounds                                      # Границы участка.
    poly = [{"lon": lon0, "lat": lat0}, {"lon": lon1, "lat": lat0}, {"lon": lon1, "lat": lat1},
            {"lon": lon0, "lat": lat1}, {"lon": lon0, "lat": lat0}]      # Замкнутый контур участка.
    parms = {"poly": poly,                                               # Область запроса.
             "srt": icesat2.SRT_LAND,                                    # Тип поверхности: суша.
             "cnf": icesat2.CNF_POSSIBLE_TEP,                            # Фотоны всех уровней достоверности: шум отсеет ATL08.
             "len": 100.0, "res": 100.0,                                 # Сегменты длиной 100 м с шагом 100 м.
             "atl08_class": ["atl08_ground", "atl08_canopy", "atl08_top_of_canopy"],   # Только фотоны земли и растительности.
             "phoreal": {"binsize": 1.0, "geoloc": "center", "use_abs_h": False, "send_waveform": False},   # Расчёт высот земли и полога.
             "t0": f"{t0}T00:00:00Z", "t1": f"{t1}T23:59:59Z"}            # Период измерений.
    failed = set()                                                       # Файлы-гранулы, которые сервер не смог прочитать.
    def on_alert(rec, session):                                          # Сообщения сервера о сбоях чтения.
        m = re.search(r"(ATL03_\S+?\.h5)", rec["text"])                  # Имя гранулы в тексте сообщения.
        if rec["code"] < 0 and m:                                        # Сбой чтения конкретной гранулы:
            failed.add(m.group(1))                                       # запоминаем её для повтора.
    for attempt in range(4):                                             # Сервер иногда временно недоступен (ошибка 502):
        try:
            first = icesat2.atl08p(parms, callbacks={"exceptrec": on_alert})   # Основной запрос.
        except Exception as e:
            first = pd.DataFrame()                                       # Пустой результат – повторим запрос.
            print("Ошибка сервиса SlideRule:", str(e)[:120])
        if len(first) > 0:                                               # Данные получены:
            break                                                        # Выходим из цикла повторов.
        if attempt < 3:                                                  # Остались попытки:
            print(f"Сервис вернул пустой ответ, повторный запрос через {60 * (attempt + 1)} с")
            time.sleep(60 * (attempt + 1))                               # Пауза перед повтором растёт с каждой попыткой.
    if len(first) == 0:                                                  # Все четыре попытки неудачны.
        raise RuntimeError("SlideRule не вернул данных. Проверьте границы участка и период или повторите запуск ячейки позже.")
    parts = [first]                                                      # Части результата.
    if failed:                                                           # Повторный запрос только для сбойных гранул.
        retry = sorted(failed)                                           # Список сбойных гранул.
        failed.clear()                                                   # Очищаем, чтобы учесть новые сбои.
        parts.append(icesat2.atl08p(parms, callbacks={"exceptrec": on_alert}, resources=retry))  # Запрашиваем только их.
        print(f"Гранул со сбоем чтения: {len(retry)}; после повторного запроса не прочитано: {len(failed)}")
    g = pd.concat([p.reset_index() for p in parts if len(p) > 0]).drop_duplicates(subset=["time", "spot"])   # Без повторов.
    df = pd.DataFrame({"time": pd.to_datetime(g["time"]),                # Время измерения.
                       "lon": g.geometry.x, "lat": g.geometry.y,         # Координаты центра сегмента.
                       "h_ground": g["h_te_median"],                     # Высота земли над эллипсоидом, м.
                       "gnd_ph_count": g["gnd_ph_count"],                # Число фотонов земли в сегменте.
                       "snowcover": g["snowcover"],                      # Флаг снега (1 – без снега, 2 – снег).
                       "landcover": g["landcover"],                      # Класс покрова по данным ATL08.
                       "h_canopy": g["h_canopy"],                        # Высота растительности над землёй, м.
                       "rgt": g["rgt"], "cycle": g["cycle"], "spot": g["spot"]})   # Номер трека, цикла и луча.
    if cache:
        df.to_parquet(cache)                                             # Сохраняем, чтобы не загружать повторно.
    return df

def ellipsoid_to_geoid(h, lon, lat, geoid="EGM2008"):
    """Пересчёт эллипсоидальной высоты в высоту над геоидом EGM2008 или EGM96."""
    scalar = np.ndim(h) == 0 and np.ndim(lon) == 0                       # Одна точка или массив точек.
    h, lon, lat = (np.asarray(v, dtype=float) for v in (h, lon, lat))
    prep = os.path.join("ggmplus_cache", f"{roi_key(ROI_BOUNDS)}.npz")  # Подготовленные данные участка из списка зон.
    os.makedirs("ggmplus_cache", exist_ok=True)
    if os.path.exists(prep) and "nlats" not in np.load(prep).files:    # Файл прежней версии без высот геоида:
        os.remove(prep)                                                  # скачиваем заново.
    if (os.path.exists(prep) or fetch_prepared("ggmplus", "npz", prep)) and "nlats" in np.load(prep).files:
        P = np.load(prep)                                                # Высоты геоида с шагом 0,01° вокруг участка.
        fi, fj = (lat - P["nlats"][0]) / 0.01, (lon - P["nlons"][0]) / 0.01   # Дробные номера узлов.
        if fi.min() >= 0 and fj.min() >= 0 and fi.max() <= len(P["nlats"]) - 1 and fj.max() <= len(P["nlons"]) - 1:
            from scipy.ndimage import map_coordinates
            N = map_coordinates(P["geoid_" + geoid.lower()], [np.atleast_1d(fi), np.atleast_1d(fj)], order=1)   # Высота геоида N.
            H = np.atleast_1d(h) - N                                     # H = h − N.
            return float(H[0]) if scalar else H.reshape(h.shape)
    epsg = {"EGM2008": "EPSG:4326+3855", "EGM96": "EPSG:4326+5773"}[geoid]   # Широта, долгота и высота над выбранным геоидом.
    to_geoid = Transformer.from_crs("EPSG:4979", epsg, always_xy=True)   # Из WGS 84 с эллипсоидальной высотой.
    H = to_geoid.transform(lon, lat, h)[2]                               # Третья координата – высота над геоидом.
    if np.allclose(H, h):                                                # Сетка геоида не загрузилась: pyproj вернул h без изменений.
        raise RuntimeError("Сетка геоида " + geoid + " не загрузилась с cdn.proj.org; перезапустите ячейку через минуту")
    return float(H) if scalar else H

GGM_SPEC = {"dg": (">i2", -2**15, 0.1, "dg"),                           # Возмущения: двухбайтовые целые, шаг 0,1 мГал.
            "geoid": (">i4", -2**31, 0.001, "ha")}                      # Квазигеоид: четырёхбайтовые целые, шаг 1 мм.

def load_ggmplus(bounds, quantity="dg", cache="ggmplus_cache"):
    """Сетка GGMplus (шаг 0,002°) по границам участка: dg – возмущение силы тяжести, мГал; geoid – высота квазигеоида, м."""
    lon0, lat0, lon1, lat1 = bounds                                      # Границы участка.
    if max(abs(lat0), abs(lat1)) > 60:                                   # Модель покрывает сушу только до ±60°.
        raise ValueError("GGMplus покрывает сушу только между 60° ю. ш. и 60° с. ш.")
    dtype, nodata, scale, suf = GGM_SPEC[quantity]                      # Формат файлов выбранной величины.
    step = 0.002                                                         # Шаг сетки GGMplus, градусы (около 220 м).
    os.makedirs(cache, exist_ok=True)                                    # Папка для скачанных тайлов.
    lats = np.arange(math.floor(lat0 / step) * step + step / 2, lat1 + step, step)   # Центры ячеек по широте.
    lons = np.arange(math.floor(lon0 / step) * step + step / 2, lon1 + step, step)   # Центры ячеек по долготе.
    prep = os.path.join(cache, f"{roi_key(ROI_BOUNDS)}.npz")             # Подготовленный фрагмент поля для участка.
    if os.path.exists(prep) or fetch_prepared("ggmplus", "npz", prep):   # Участок из списка зон:
        P = np.load(prep)                                                # поле вокруг участка с запасом 0,1°.
        ii = np.round((lats - P["lats"][0]) / step).astype(int)          # Строки фрагмента для наших широт.
        jj = np.round((lons - P["lons"][0]) / step).astype(int)          # Столбцы для наших долгот.
        if ii.min() >= 0 and jj.min() >= 0 and ii.max() < len(P["lats"]) and jj.max() < len(P["lons"]):
            return P[quantity][np.ix_(ii, jj)], lons, lats              # Те же значения, что и из тайлов модели.
    out = np.full((len(lats), len(lons)), np.nan, np.float32)            # Результат для участка.
    for t_lat in range(int(math.floor(lat0 / 5) * 5), int(math.ceil(lat1 / 5) * 5), 5):       # Тайлы 5°×5°,
        for t_lon in range(int(math.floor(lon0 / 5) * 5), int(math.ceil(lon1 / 5) * 5), 5):   # покрывающие участок.
            name = f"{'N' if t_lat >= 0 else 'S'}{abs(t_lat):02d}{'E' if t_lon >= 0 else 'W'}{abs(t_lon):03d}"   # Например, N55E090.
            path = os.path.join(cache, f"{name}.{suf}")                  # Файл тайла в кэше.
            if not os.path.exists(path):                                 # Скачиваем только новые тайлы:
                url = f"https://ddfe.curtin.edu.au/models/GGMplus/data/{quantity}/{name}.{suf}"  # Адрес тайла на сервере.
                req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})   # Сервер отклоняет запросы без имени браузера.
                try:
                    with urllib.request.urlopen(req, timeout=120) as r, open(path + ".part", "wb") as f:
                        f.write(r.read())                                # сначала во временный файл,
                except urllib.error.HTTPError as e:                      # Сервер отказал (в Colab бывает 403):
                    raise RuntimeError(f"Сервер GGMplus отказал ({e.code}). Скачайте файл {name}.{suf} по адресу {url} "
                                       f"и загрузите его в папку {cache} (панель «Файлы» слева), затем выполните ячейку ещё раз.") from None
                os.replace(path + ".part", path)                         # затем переименовываем целиком.
            a = np.fromfile(path, dtype=dtype).astype(np.float32)        # Двоичные целые числа тайла.
            a[a == nodata] = np.nan                                      # Пропуски (море) – NaN.
            a = a.reshape(2500, 2500).T * scale                          # Записи идут вдоль меридианов; переводим в мГал или м.
            ti = np.round((lats - (t_lat + 0.001)) / step).astype(int)   # Номера строк тайла для наших широт.
            tj = np.round((lons - (t_lon + 0.001)) / step).astype(int)   # Номера столбцов для наших долгот.
            mi, mj = (ti >= 0) & (ti < 2500), (tj >= 0) & (tj < 2500)    # Какие ячейки участка попадают в этот тайл.
            out[np.ix_(mi, mj)] = a[np.ix_(ti[mi], tj[mj])]              # Переносим значения в результат.
    return out, lons, lats

def load_photons(parms, cache):
    """Фотоны ATL03 отрезка трека: подготовленные данные для участков из списка зон, иначе сервис SlideRule."""
    import geopandas as gpd
    if os.path.exists(cache) or fetch_prepared("photons", "parquet", cache):   # Подготовленные фотоны:
        df = pd.read_parquet(cache).set_index("time")                   # таблица с координатами точек.
        return gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326")
    from sliderule import sliderule, icesat2                             # Другой участок: запрос к сервису.
    sliderule.init("slideruleearth.io", verbose=False)                   # Подключение к открытому серверу.
    return icesat2.atl03sp(parms)                                        # Фотоны ATL03 с классами ATL08.

In [ ]:
# Готовые функции: признаки рельефа, координаты, блоки, метрики, классы покрова
def relief_features(dem, res=30):
    """Уклон, индексы топографического положения TPI и шероховатость по модели рельефа."""
    gy, gx = np.gradient(dem, res, res)                                  # Изменение высоты на метр по осям.
    m7, m31 = uniform_filter(dem, 7), uniform_filter(dem, 31)            # Средняя высота в окнах 7×7 (210 м) и 31×31 (930 м).
    slope = np.degrees(np.arctan(np.hypot(gx, gy)))                      # Уклон, градусы.
    rough7 = np.sqrt(np.maximum(uniform_filter(dem * dem, 7) - m7 ** 2, 0))   # Стандартное отклонение высоты в окне 7×7.
    return {"slope": slope, "tpi7": dem - m7, "tpi31": dem - m31, "rough7": rough7}   # TPI > 0 – возвышение, < 0 – понижение.

def lonlat_to_rowcol(lon, lat, grid):
    """Дробные индексы строки и столбца сетки участка для точек."""
    X, Y = Transformer.from_crs("EPSG:4326", grid["crs"], always_xy=True).transform(lon, lat)   # Градусы → метры UTM.
    row = (grid["y1"] - np.asarray(Y)) / grid["res"] - 0.5               # Строки отсчитываются от северного края.
    col = (np.asarray(X) - grid["x0"]) / grid["res"] - 0.5               # Столбцы – от западного края.
    return row, col

def grid_lonlat(grid):
    """Долгота и широта центров всех ячеек сетки."""
    rr, cc = np.indices((grid["H"], grid["W"]))                          # Номера строки и столбца каждой ячейки.
    X = grid["x0"] + (cc + 0.5) * grid["res"]                            # Координаты центров в метрах UTM.
    Y = grid["y1"] - (rr + 0.5) * grid["res"]                            # Север сверху: Y убывает по строкам.
    return Transformer.from_crs(grid["crs"], "EPSG:4326", always_xy=True).transform(X, Y)   # Метры → градусы.

def block_id(row, col, grid, size_m):
    """Номер пространственного блока размером size_m для пространственной проверки."""
    k = size_m / grid["res"]                                             # Сторона блока в ячейках.
    block_row = np.floor(np.asarray(row) / k)                            # Номер блока по вертикали.
    block_col = np.floor(np.asarray(col) / k)                            # Номер блока по горизонтали.
    return (block_row * 10000 + block_col).astype(int)                   # Один номер на блок.

def height_metrics(e):
    """Метрики ошибки высот: RMSE, MAE, медианное смещение, NMAD, LE90."""
    e = np.asarray(e)                                                    # Ошибки как массив NumPy.
    return {"RMSE": np.sqrt(np.mean(e ** 2)),                            # Среднеквадратическая ошибка.
            "MAE": np.mean(np.abs(e)),                                   # Средняя абсолютная ошибка.
            "смещение": np.median(e),                                    # Систематическое завышение (+) или занижение (−).
            "NMAD": 1.4826 * np.median(np.abs(e - np.median(e))),        # Устойчивый аналог стандартного отклонения.
            "LE90": np.percentile(np.abs(e), 90)}                        # Ошибка, которую не превышают 90 % точек.

CLASS_NAMES = {1: "Лес", 2: "Кустарник", 3: "Луга и пашня", 4: "Застройка", 5: "Открытый грунт", 6: "Вода", 7: "Болота и мхи"}  # Классы покрова, используемые в работе.
CLASS_COLORS = {1: "#1f7a3a", 2: "#9bbf4a", 3: "#e8d36b", 4: "#c0392b", 5: "#b8a48a", 6: "#2e86de", 7: "#7fc8b8"}  # Цвета классов на картах.
WC_TO_CLASS = {10: 1, 20: 2, 30: 3, 40: 3, 50: 4, 60: 5, 80: 6, 90: 7, 95: 7, 100: 7}   # Код WorldCover → класс покрова в работе.

In [ ]:
# Готовые функции: трёхмерная модель местности
def mesh_trace(height, colors_rgb, grid, step, exag):
    """Треугольная сетка рельефа с цветами вершин; возвращает объект plotly и пропорции сцены."""
    Z = height[::step, ::step]                                           # Прореживаем сетку: каждая step-я ячейка.
    C = colors_rgb[::step, ::step]                                       # Цвета тех же ячеек.
    h, w = Z.shape                                                       # Число строк и столбцов вершин.
    xs = np.arange(w) * grid["res"] * step / 1000                        # Координаты вершин по X, км.
    ys = (h - 1 - np.arange(h)) * grid["res"] * step / 1000              # По Y, км (север сверху).
    XX, YY = np.meshgrid(xs, ys)                                         # Координаты всех вершин.
    idx = np.arange(h * w).reshape(h, w)                                 # Номер каждой вершины.
    a, b = idx[:-1, :-1].ravel(), idx[:-1, 1:].ravel()                   # Верхние углы каждой клетки сетки.
    c, d = idx[1:, :-1].ravel(), idx[1:, 1:].ravel()                     # Нижние углы.
    i = np.r_[a, b].astype(np.int32)                                     # Клетка делится на два треугольника,
    j = np.r_[b, d].astype(np.int32)                                     # (a, b, c) и (b, d, c);
    k = np.r_[c, c].astype(np.int32)                                     # i, j, k – номера их вершин.
    colors = [f"rgb({r},{g},{bl})" for r, g, bl in C.reshape(-1, 3)]     # Цвет каждой вершины.
    tr = go.Mesh3d(x=XX.ravel().astype(np.float32), y=YY.ravel().astype(np.float32), z=Z.ravel().astype(np.float32),  # Поверхность из треугольников.
                   i=i, j=j, k=k, vertexcolor=colors, flatshading=False, hoverinfo="skip", showlegend=False,
                   lighting=dict(ambient=0.55, diffuse=0.9, specular=0.05))   # Мягкое освещение рельефа.
    zr = float(np.nanmax(Z) - np.nanmin(Z)) * exag / 1000                # Перепад высот с вертикальным масштабом, км.
    return tr, dict(x=1, y=h / w, z=max(zr / max(xs.max(), 1e-6), 0.05))   # Пропорции сцены по осям.

SYNC_JS = """var gd = document.getElementById('{plot_id}');
gd.on('plotly_relayout', function (e) {
  if (gd._sync) return;
  var cam = e['scene.camera'] || e['scene2.camera'];
  if (!cam) return;
  gd._sync = true;
  Plotly.relayout(gd, {'scene.camera': cam, 'scene2.camera': cam}).then(function () { gd._sync = false; });
});"""   # Скрипт для браузера: поворот одной сцены повторяется на второй.

def mesh3d_pair(height, rgb8, cls_rgb, classes, grid, step=4, exag=3, title=""):
    """Два трёхмерных вида в одной строке: снимок и карта покрова на одной модели рельефа."""
    from plotly.subplots import make_subplots
    fig = make_subplots(rows=1, cols=2, specs=[[{"type": "scene"}, {"type": "scene"}]], horizontal_spacing=0.01,  # Две трёхмерные сцены рядом.
                        subplot_titles=["Снимок Sentinel-2 на уточнённом рельефе", "Карта покрова (модель ИИ) на уточнённом рельефе"])
    t1, aspect = mesh_trace(height, rgb8, grid, step, exag)              # Рельеф в цветах снимка.
    t2, _ = mesh_trace(height, cls_rgb, grid, step, exag)                # Тот же рельеф в цветах классов.
    fig.add_trace(t1, 1, 1)                                              # Левая сцена.
    fig.add_trace(t2, 1, 2)                                              # Правая сцена.
    for c in classes:                                                    # Легенда классов (невидимые точки).
        fig.add_trace(go.Scatter3d(x=[None], y=[None], z=[None], mode="markers", name=CLASS_NAMES[c],
                                   marker=dict(size=10, symbol="square", color=CLASS_COLORS[c])), 1, 2)
    cam = dict(eye=dict(x=0.08, y=-1.14, z=0.66), center=dict(x=0, y=0, z=-0.12))   # Обзор с юга под углом сверху.
    hidden = dict(visible=False, showbackground=False)                   # Без осей и фона.
    scene = dict(aspectmode="manual", aspectratio=aspect, camera=cam, xaxis=hidden, yaxis=hidden, zaxis=hidden)  # Одинаковые настройки обеих сцен.
    fig.update_layout(title=dict(text=title, y=0.98, x=0.5, xanchor="center"), height=620, margin=dict(l=0, r=0, t=90, b=0),  # Заголовок, размер, легенда.
                      scene=scene, scene2=scene,
                      legend=dict(orientation="h", yanchor="bottom", y=0.01, xanchor="center", x=0.75, font=dict(size=14)))
    for a in fig.layout.annotations:                                     # Подписи панелей ниже общего заголовка.
        a.y = 0.93
        a.font = dict(size=15)
    return fig

def surface_3d(ax, Z, colors=None, cmap="terrain", exag=3, z_ratio=None, title="", zlabel="Высота, м", nodes=170):
    """Трёхмерная поверхность растра в осях matplotlib: высота – значения массива, цвет – снимок или палитра."""
    step = max(1, max(Z.shape) // nodes)                                 # Прореживаем: не больше nodes узлов по стороне.
    Zs = Z[::step, ::step]                                               # Значения в узлах поверхности.
    h, w = Zs.shape                                                      # Число узлов по строкам и столбцам.
    X, Y = np.meshgrid(np.arange(w) * GRID["res"] * step / 1000,         # Координаты узлов, км:
                       (h - 1 - np.arange(h)) * GRID["res"] * step / 1000)   # север – дальняя сторона.
    if colors is None:                                                   # Цвет по значению (палитра cmap):
        lo, hi = np.nanpercentile(Zs, 2), np.nanpercentile(Zs, 98)       # Растяжение без крайних 2 %.
        colors = plt.get_cmap(cmap)(np.clip((Z - lo) / (hi - lo), 0, 1))
    ax.plot_surface(X, Y, Zs, facecolors=colors[::step, ::step], rstride=1, cstride=1,
                    linewidth=0, antialiased=False, shade=False)          # Поверхность из четырёхугольников.
    if z_ratio is None:                                                  # Для рельефа – вертикальный масштаб exag:
        z_ratio = exag * float(np.nanmax(Zs) - np.nanmin(Zs)) / 1000 / X.max()
    ax.set_box_aspect((1, Y.max() / X.max(), max(z_ratio, 0.05)))        # Пропорции осей.
    ax.view_init(elev=38, azim=-115)                                     # Взгляд с юго-запада под углом сверху.
    ax.zaxis.set_major_locator(plt.MaxNLocator(3))                       # Три деления по оси высоты: подписи не налезают.
    ax.set_xlabel("км"); ax.set_ylabel("км"); ax.set_zlabel(zlabel)      # Подписи осей.
    ax.set_title(title)

def overlay_layer(arr, name, cmap, vmin, vmax, categorical=False, opacity=0.7, show=False):
    """Растр сетки участка – слой folium в географических координатах (пересчёт из UTM в широту и долготу)."""
    from rasterio.warp import reproject, Resampling, calculate_default_transform
    from rasterio.transform import from_origin
    left, top = GRID["x0"], GRID["y1"]                                   # Левый верхний угол сетки, м.
    right, bottom = left + GRID["W"] * GRID["res"], top - GRID["H"] * GRID["res"]
    src_t = from_origin(left, top, GRID["res"], GRID["res"])             # Привязка исходной сетки.
    dst_t, w, h = calculate_default_transform(GRID["crs"], "EPSG:4326", GRID["W"], GRID["H"], left, bottom, right, top)
    out = np.full((h, w), np.nan, np.float32)                            # Растр в градусах.
    reproject(arr.astype(np.float32), out, src_transform=src_t, src_crs=GRID["crs"], dst_transform=dst_t, dst_crs="EPSG:4326",
              resampling=Resampling.nearest if categorical else Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
    rgba = plt.get_cmap(cmap)(np.clip((out - vmin) / (vmax - vmin), 0, 1)) if not categorical else cmap(out.astype(int))
    rgba[np.isnan(out)] = 0                                              # Пропуски – прозрачные.
    west, north = dst_t.c, dst_t.f                                       # Границы растра в градусах.
    east, south = west + w * dst_t.a, north + h * dst_t.e
    return folium.raster_layers.ImageOverlay((rgba * 255).astype(np.uint8), bounds=[[south, west], [north, east]],
                                             opacity=opacity, name=name, show=show)

In [ ]:
GRID = make_grid(ROI_BOUNDS)                                             # Рабочая сетка участка, 30 м, UTM.
print(f"Сетка {GRID['W']}×{GRID['H']} пикселей по 30 м, {GRID['crs']}")

---

## **4. Изображения участка**

### **4.1. Безоблачный композит и остальные изображения на сетке участка**

> 📚 **Подсказка:** лекция «Цифровая обработка спутниковых изображений рельефа и гравитационного поля», **разделы 3.1–3.2**.

Ячейка загружает композит Sentinel-2, четыре модели рельефа, карту покрова, высоту леса и год потери леса на сетку 30 м. Это занимает около минуты.

In [ ]:
t = time.time()                                                          # Засекаем время загрузки.
s2_img, n_scenes = s2_composite(REGION, *S2_PERIOD)                      # Композит Sentinel-2 и число снимков.
all_layers = s2_img.addBands(dem_images(REGION)).addBands(auxiliary_images())   # Все слои одним изображением.
stack = gee_to_numpy(all_layers, GRID)                                   # Словарь {слой: массив на сетке участка}.
print(f"Каналов получено: {len(stack)}; время {time.time() - t:.0f} с")  # Сколько слоёв получено и за какое время.
s2_filter = pd.DataFrame([                                               # Таблица шагов фильтрации снимков.
    ["Отбор по территории и периоду", f"{S2_PERIOD[0]} – {S2_PERIOD[1]}", f"{n_scenes} снимков"],
    ["Маска облаков и теней Cloud Score+", "cs_cdf ≥ 0,6", f"медиана {np.nanmedian(stack['n_clear']):.0f} безоблачных наблюдений на пиксель (минимум {np.nanmin(stack['n_clear']):.0f})"],
    ["Медианный композит", "медиана по каждому пикселю", f"пикселей без данных: {100 * np.isnan(stack['B4']).mean():.2f}%"],
], columns=["Шаг фильтрации", "Правило", "Результат"])
s2_filter                                                                # Показываем таблицу.

In [ ]:
rgb = np.stack([stack["B4"], stack["B3"], stack["B2"]], -1)              # Красный, зелёный, синий каналы.
rgb = np.clip(rgb / 0.18, 0, 1) ** (1 / 1.25)                           # Растяжение яркости и осветление для показа.
ls = LightSource(315, 40)                                                # Источник света: азимут 315°, высота 40°.
fig, ax = plt.subplots(1, 2, figsize=(13, 6))                            # Два рисунка рядом.
ax[0].imshow(rgb); ax[0].set_title("Sentinel-2, безоблачный композит")   # Слева – композит.
ax[1].imshow(ls.hillshade(stack["cop"], vert_exag=3, dx=30, dy=30), cmap="gray"); ax[1].set_title("Copernicus DEM, отмывка рельефа")  # Справа – отмывка.
for a in ax: a.axis("off")                                               # Без осей координат.
plt.tight_layout(); plt.show()               # Показываем рисунок.

**Вопрос:** Сравните композит и отмывку рельефа: какие объекты территории видны на обоих изображениях, а какие только на одном? Есть ли на композите остатки облаков или теней?

**Ответ:** _______________

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1.4, 1]})   # Карта и гистограмма.
im = ax[0].imshow(stack["n_clear"], cmap="viridis"); ax[0].set_title("Безоблачных наблюдений на пиксель"); ax[0].axis("off"); plt.colorbar(im, ax=ax[0], fraction=0.046)
ax[1].hist(stack["n_clear"].ravel(), bins=30, color="#2e86de"); ax[1].set_xlabel("Наблюдений"); ax[1].set_ylabel("Пикселей"); ax[1].set_title("Распределение по участку")
plt.tight_layout(); plt.show()                # Показываем рисунок.

**Вопрос:** Сколько безоблачных наблюдений приходится на пиксель (медиана, минимум) и где их меньше всего? Достаточно ли этого для устойчивой медианы?

**Ответ:** _______________

### **4.2. Спектральные индексы своими руками**

**Задание 4.2.** Рассчитайте индексы NDVI, NDWI и NDBI по каналам композита `stack` как нормализованные разности (раздел 3.4 лекции) и сравните с индексами, рассчитанными в Earth Engine: наибольшее расхождение должно быть порядка $10^{-6}$ (точность чисел float32).

In [ ]:
b = {k: stack[k] for k in ["B3", "B4", "B8", "B11"]}                     # Нужные каналы композита.
my_ndvi = ...                                                            # NDVI = (B8 − B4) / (B8 + B4).
my_ndwi = ...                                                            # NDWI = (B3 − B8) / (B3 + B8).
my_ndbi = ...                                                            # NDBI = (B11 − B8) / (B11 + B8).
for name, mine in [("NDVI", my_ndvi), ("NDWI", my_ndwi), ("NDBI", my_ndbi)]:
    print(f"{name}: наибольшее расхождение с GEE {np.nanmax(np.abs(mine - stack[name])):.6f}")

Распределения индексов по классам WorldCover (готовая ячейка):

In [ ]:
label_map = np.zeros(stack["wc"].shape, int)                             # Классы покрова работы (1–7); 0 – класс не используется.
for wc_code, our_class in WC_TO_CLASS.items():                           # Коды WorldCover → классы работы.
    label_map[stack["wc"] == wc_code] = our_class
classes_present = [c for c in CLASS_NAMES if (label_map == c).sum() >= 200]   # Классы, которых на участке достаточно.
fig, ax = plt.subplots(1, 3, figsize=(16, 5))                            # Три карты индексов.
for a, (name, cmap, lim) in zip(ax, [("NDVI", "YlGn", (-0.2, 0.9)), ("NDWI", "Blues", (-0.6, 0.4)), ("NDBI", "copper", (-0.5, 0.3))]):
    im = a.imshow(stack[name], cmap=cmap, vmin=lim[0], vmax=lim[1]); a.set_title(name); a.axis("off"); plt.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.show()                         # Показываем рисунок.
rng = np.random.default_rng(SEED)                                        # Генератор случайных чисел.
sample = rng.choice(label_map.size, 60000, replace=False)                # Выборка ячеек для диаграмм.
sample = sample[label_map.ravel()[sample] > 0]                           # Только ячейки с классом.
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))                          # Распределения индексов по классам.
rows = []                                                                # Строки таблицы медиан.
for a, name in zip(ax, ["NDVI", "NDWI", "NDBI"]):
    vals = stack[name].ravel()[sample]; labs = label_map.ravel()[sample]
    a.boxplot([vals[labs == c] for c in classes_present], tick_labels=[CLASS_NAMES[c] for c in classes_present], showfliers=False)
    a.set_title(f"{name} по классам WorldCover"); a.tick_params(axis="x", rotation=30)
for c in classes_present:                                                # Медианы индексов по классам.
    m_ = label_map == c
    rows.append([CLASS_NAMES[c], int(m_.sum())] + [np.nanmedian(stack[n][m_]) for n in ["NDVI", "NDWI", "NDBI"]])
idx_by_class = pd.DataFrame(rows, columns=["Класс", "Ячеек", "NDVI, медиана", "NDWI, медиана", "NDBI, медиана"]).round(2)
plt.tight_layout(); plt.show()              # Показываем рисунок.
idx_by_class                                                             # Показываем таблицу.

**Вопрос:** Какие классы покрова на вашей территории хорошо разделяются каждым индексом, а какие перекрываются? Приведите медианы индексов для двух-трёх классов из таблицы.

**Ответ:** _______________

### **4.3. Как период съёмки меняет композит**

**Задание 4.3.** Добавьте в список `variants` свой вариант композита — другой период (например, сентябрь или июнь – июль) или другой порог ясности — и выполните сравнение.

In [ ]:
PREVIEW = make_grid(ROI_BOUNDS, res=120)                                 # Грубая сетка для быстрых сравнений.
def show_rgb(d, scale=1.0):
    """Три канала композита в изображение для показа."""
    return np.clip(np.stack([d["B4"], d["B3"], d["B2"]], -1) / scale / 0.18, 0, 1) ** (1 / 1.25)
year = S2_PERIOD[0][:4]                                                  # Год выбранного периода.
variants = [("лето, порог 0,6 (принято)", S2_PERIOD, 0.6), ("лето, порог 0,5", S2_PERIOD, 0.5), ("лето, порог 0,7", S2_PERIOD, 0.7),
            ("апрель – май", (f"{year}-04-01", f"{year}-05-31"), 0.6), ("весь год", (f"{year}-01-01", f"{year}-12-31"), 0.6)]   # Варианты.
variants.append(...)   # Ваш код: свой вариант (название, (начало, конец), порог)
rows, previews = [], []                                                  # Строки таблицы и изображения.
for name, period, thr in variants:                                       # Для каждого варианта:
    img, n = s2_composite(REGION, *period, cs_threshold=thr)             # Композит на сервере.
    d = gee_to_numpy(img.select(["B4", "B3", "B2", "n_clear"]), PREVIEW)   # Каналы и число наблюдений на сетке 120 м.
    rows.append([name, f"{period[0]} – {period[1]}", thr, n, np.nanmedian(d["n_clear"]), 100 * np.isnan(d["B4"]).mean()])
    previews.append((name, d))
comp_var = pd.DataFrame(rows, columns=["Вариант", "Период", "Порог ясности", "Снимков", "Безоблачных наблюдений на пиксель, медиана", "Пикселей без данных, %"]).round(1)
fig, ax = plt.subplots(1, len(previews), figsize=(18, 4.3))              # Композиты в ряд.
for a, (name, d) in zip(ax, previews):
    a.imshow(show_rgb(d)); a.set_title(name, fontsize=10); a.axis("off")
plt.tight_layout(); plt.show()              # Показываем рисунок.
comp_var                                                                 # Показываем таблицу.

**Вопрос:** Как период съёмки и порог ясности изменили число наблюдений на пиксель и вид композита? Какой период вы выбрали бы для карты покрова своей территории и почему?

**Ответ:** _______________

---

## **5. Рельеф как изображение**

> 📚 **Подсказка:** лекция «Цифровая обработка спутниковых изображений рельефа и гравитационного поля», **раздел 4**.

In [ ]:
dem = stack["cop"].astype(np.float64)                                               # Уклон, TPI, шероховатость.                                    # Изображение рельефа Copernicus: значение пикселя – высота, м.

### **5.1. Признаки рельефа фильтрами в окне**

**Задание 5.1.** Допишите функцию `my_relief_features(z, res)`: уклон (через `np.gradient`), TPI в окнах 7×7 и 31×31 (высота минус `uniform_filter`), шероховатость в окне 7×7 (стандартное отклонение через средние $z$ и $z^2$). Сравните результат с готовой функцией `relief_features`: расхождение должно быть нулевым или порядка ошибок округления.

In [ ]:
def my_relief_features(z, res=30):
    """Уклон, TPI в окнах 7×7 и 31×31 и шероховатость в окне 7×7 по изображению рельефа z."""
    gy, gx = ...                                                         # Производные: np.gradient(z, res, res).
    slope = ...                                                          # Уклон, градусы: arctan модуля градиента.
    m7, m31 = ...                                                        # Средние в окнах: uniform_filter(z, 7) и uniform_filter(z, 31).
    tpi7, tpi31 = ...                                                    # Высота минус среднее в окне.
    rough7 = ...                                                         # Стандартное отклонение в окне 7×7: sqrt(mean(z²) − mean(z)²).
    return {"slope": slope, "tpi7": tpi7, "tpi31": tpi31, "rough7": rough7}

mine = my_relief_features(dem)
REL = relief_features(dem)                                               # Готовая функция для сравнения.
for k in REL:
    print(f"{k}: наибольшее расхождение {np.nanmax(np.abs(mine[k] - REL[k])):.2e}")

In [ ]:
REL = relief_features(dem)                                               # Уклон, TPI, шероховатость.
fig, ax = plt.subplots(1, 4, figsize=(18, 4.6))                          # Четыре карты признаков.
for a, (k, t, cmap, lim) in zip(ax, [("slope", "Уклон, °", "magma", (0, 30)), ("tpi7", "TPI 7×7 (210 м), м", "RdBu_r", (-15, 15)),
                                     ("tpi31", "TPI 31×31 (930 м), м", "RdBu_r", (-60, 60)), ("rough7", "Шероховатость 7×7, м", "viridis", (0, 15))]):
    im = a.imshow(REL[k], cmap=cmap, vmin=lim[0], vmax=lim[1]); a.set_title(t); a.axis("off"); plt.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.show()                # Показываем рисунок.
feat_tbl = pd.DataFrame({"высота": dem.ravel(), **{k: v.ravel() for k, v in REL.items()}, "NDVI": stack["NDVI"].ravel(), "высота леса ETH": stack["eth"].ravel()})
corr_tbl = feat_tbl.sample(50000, random_state=SEED).corr().round(2)    # Корреляции признаков по выборке ячеек.
corr_tbl                                                                 # Показываем таблицу.

**Вопрос:** Какие признаки рельефа на вашей территории сильно коррелируют между собой (приведите коэффициенты)? Почему уклон и шероховатость почти взаимозаменяемы?

**Ответ:** _______________

### **5.2. Фильтры низких и высоких частот**

**Задание 5.2.** Выберите размер окна `WINDOW` (нечётное число пикселей), получите низкие частоты рельефа сглаживанием `uniform_filter` и высокие частоты как разность исходного и сглаженного изображений. Покажите три изображения рядом и выведите стандартное отклонение высоких частот. Попробуйте 2–3 разных окна.

In [ ]:
WINDOW = ...                                                             # Размер окна фильтра, пиксели (нечётное число, например 21).
low = ...                                                                # Низкие частоты: uniform_filter(dem, WINDOW).
high = ...                                                               # Высокие частоты: dem − low.
# Ваш код: три изображения рядом (исходное, low, high) и стандартное отклонение high

**Вопрос:** Какое окно вы выбрали и что видно на изображении высоких частот (гребни, долины, бровки)? Как меняется стандартное отклонение высоких частот при увеличении окна и почему?

**Ответ:** _______________

### **5.3. Трёхмерные поверхности**

**Задание 5.3.** Постройте рядом две трёхмерные поверхности функцией `surface_3d`: рельеф с текстурой снимка (готово) и изображение высоких частот `high`.

In [ ]:
fig = plt.figure(figsize=(17, 7.5))
ax = fig.add_subplot(1, 2, 1, projection="3d")
surface_3d(ax, dem, colors=rgb, title="Рельеф, цвет – снимок Sentinel-2 (×3)")
# Ваш код: вторая поверхность – изображение high (палитра "RdBu_r", z_ratio=0.3)
plt.tight_layout(); plt.show()

**Вопрос:** Какие формы рельефа видны на трёхмерной поверхности, но плохо различимы на плоской отмывке? Что показывает поверхность высоких частот?

**Ответ:** _______________

---

## **6. Лазерные измерения ICESat-2**

> 📚 **Подсказка:** лекция «Цифровая обработка спутниковых изображений рельефа и гравитационного поля», **разделы 2.4, 2.6 и 5**.

### **6.1. Загрузка измерений**

In [ ]:
t = time.time()                                                          # Засекаем время.
isat = load_icesat2(ROI_BOUNDS, *ICESAT_PERIOD, cache=f"{OUT}/icesat2_atl08.parquet")   # Сегменты ATL08 участка.
print(f"Сегментов ICESat-2: {len(isat)}; треков: {isat.rgt.nunique()}; период {isat.time.min():%Y-%m} – {isat.time.max():%Y-%m}; {time.time() - t:.0f} с")

In [ ]:
by_beam = isat.groupby("spot").agg(сегментов=("h_ground", "size"), фотонов_земли_медиана=("gnd_ph_count", "median"),   # Сводка по лучам.
                                   доля_с_найденной_землёй=("h_ground", lambda s: round(100 * s.notna().mean(), 1))).reset_index()
by_beam.columns = ["Луч (spot)", "Сегментов", "Фотонов земли в сегменте, медиана", "Сегментов с найденной землёй, %"]
by_beam["Мощность луча"] = np.where(by_beam["Фотонов земли в сегменте, медиана"] >= by_beam["Фотонов земли в сегменте, медиана"].median(), "сильный", "слабый")   # По числу фотонов.
by_month = isat.groupby(isat.time.dt.month).agg(сегментов=("h_ground", "size"), со_снегом=("snowcover", lambda s: round(100 * (s > 1).mean())))   # По месяцам.
by_month.index.name = "Месяц"; by_month.columns = ["Сегментов", "Со снегом, %"]
display(by_beam)                                                         # Таблица по лучам.
by_month.T                                                               # Таблица по месяцам (в строку).

In [ ]:
pr_, pc_ = lonlat_to_rowcol(isat.lon.values, isat.lat.values, GRID)   # Положение сегментов на сетке просмотра.
fig, ax = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.3, 1]})   # Карта и диаграмма.
ax[0].imshow(rgb)                                                 # Подложка – композит.
snow = isat.snowcover.values > 1                                         # Сегменты со снегом.
ax[0].scatter(pc_[~snow], pr_[~snow], s=0.4, c="#ff8c00", label="без снега"); ax[0].scatter(pc_[snow], pr_[snow], s=0.4, c="#7fd4ff", label="со снегом")
ax[0].set_xlim(0, GRID["W"]); ax[0].set_ylim(GRID["H"], 0); ax[0].axis("off"); ax[0].legend(loc="lower left", markerscale=12)
ax[0].set_title(f"Сегменты ICESat-2: {len(isat)} на {isat.rgt.nunique()} опорных треках, {isat.time.min():%Y-%m} – {isat.time.max():%Y-%m}")
ax[1].bar(by_month.index, by_month["Сегментов"], color="#2e86de"); ax[1].set_xticks(range(1, 13)); ax[1].set_xlabel("Месяц"); ax[1].set_ylabel("Сегментов")
ax[1].set_title("Число сегментов по месяцам (все годы)")
plt.tight_layout(); plt.show()                   # Показываем рисунок.

**Вопрос:** Сколько сегментов и треков дала ваша зона? Во сколько раз сильные лучи дают больше фотонов земли, чем слабые? В какие месяцы измерения сделаны по снегу и какую долю они составляют? Как расположены треки на участке и где высота земли останется неизвестной?

**Ответ:** _______________

### **6.2. Облако фотонов на одном километре трека**

In [ ]:
from sliderule import icesat2                                            # Константы запроса к сервису SlideRule.
isat["trk"] = isat.rgt.astype(str) + "_" + isat.cycle.astype(str) + "_" + isat.spot.astype(str)   # Одна линия: трек, цикл, луч.
summer = isat[isat.snowcover <= 1]                                       # Летние измерения (без снега).
line = summer[summer.trk == summer.groupby("trk").size().idxmax()].sort_values("lat")   # Самая длинная летняя линия.
piece = line.iloc[len(line) // 2 - 5: len(line) // 2 + 6]               # Одиннадцать сегментов из середины линии (1,1 км).
day = piece.time.min().strftime("%Y-%m-%d")                              # Дата пролёта.
lon0, lon1, lat0, lat1 = piece.lon.min() - 0.02, piece.lon.max() + 0.02, piece.lat.min() - 0.001, piece.lat.max() + 0.001   # Рамка отрезка.
parms = {"poly": [{"lon": lon0, "lat": lat0}, {"lon": lon1, "lat": lat0}, {"lon": lon1, "lat": lat1}, {"lon": lon0, "lat": lat1}, {"lon": lon0, "lat": lat0}],
         "srt": icesat2.SRT_LAND, "cnf": icesat2.CNF_POSSIBLE_TEP, "len": 100.0, "res": 100.0, "cnt": 1, "ats": 0.0,   # Все фотоны, сегменты 100 м.
         "atl08_class": ["atl08_noise", "atl08_ground", "atl08_canopy", "atl08_top_of_canopy", "atl08_unclassified"],   # Классы ATL08.
         "t0": f"{day}T00:00:00Z", "t1": f"{day}T23:59:59Z"}             # Только этот пролёт.
ph = load_photons(parms, cache=f"{OUT}/icesat2_atl03_отрезок.parquet")   # Фотоны ATL03 с классами ATL08.
ph = ph[ph.spot == piece.spot.iloc[0]].copy()                            # Только выбранный луч.
to_m = Transformer.from_crs("EPSG:4326", GRID["crs"], always_xy=True) if "GRID" in globals() else Transformer.from_crs("EPSG:4326", make_grid(ROI_BOUNDS)["crs"], always_xy=True)
px, py = to_m.transform(ph.geometry.x.values, ph.geometry.y.values)      # Фотоны в метрах.
sx, sy = to_m.transform(piece.lon.values, piece.lat.values)              # Центры сегментов в метрах.
v = np.array([sx[-1] - sx[0], sy[-1] - sy[0]]); v /= np.hypot(*v)        # Направление линии.
ph["dist"] = (px - sx[0]) * v[0] + (py - sy[0]) * v[1]                   # Расстояние вдоль линии от первого сегмента, м.
seg_dist = (sx - sx[0]) * v[0] + (sy - sy[0]) * v[1]                     # То же для сегментов.
classes = {0: ("шум", "#b0b0b0"), 1: ("земля", "#8b4513"), 2: ("полог", "#2e8b57"), 3: ("верх полога", "#9acd32"), 4: ("без класса", "#dddddd")}
fig, ax = plt.subplots(figsize=(14, 5))                                  # Профиль фотонов.
for code_, (name, color) in classes.items():                             # Каждый класс своим цветом.
    m_ = ph.atl08_class == code_
    if m_.any(): ax.scatter(ph.dist[m_], ph.height[m_], s=3, color=color, label=f"{name} ({int(m_.sum())})")
ax.plot(seg_dist, piece.h_ground.values, "k_", markersize=18, mew=2.5, label="высота земли сегмента ATL08")   # Высоты земли сегментов.
ax.plot(seg_dist, (piece.h_ground + piece.h_canopy.fillna(0)).values, "k^", markersize=6, label="земля + высота растительности ATL08")
top = (piece.h_ground + piece.h_canopy.fillna(0)).max()                 # Верх полога на отрезке.
ax.set_ylim(piece.h_ground.min() - 25, top + 45)                         # Показываем только окрестность поверхности: шум выше и ниже обрезан.
ax.set_xlabel("Расстояние вдоль трека, м"); ax.set_ylabel("Высота над эллипсоидом, м"); ax.legend(loc="best", fontsize=9, markerscale=3, framealpha=0.9)
ax.set_title(f"Фотоны ATL03 на отрезке трека {piece.rgt.iloc[0]}, цикл {piece.cycle.iloc[0]}, луч {piece.spot.iloc[0]}, {day}")
plt.show()                                              # Показываем рисунок.
ph_tbl = ph.groupby("atl08_class").size().rename("Фотонов").to_frame(); ph_tbl.index = [classes[c][0] for c in ph_tbl.index]
print(f"Фотонов земли на сегмент: {int((ph.atl08_class == 1).sum()) / len(piece):.0f}; медиана числа фотонов земли по таблице ATL08: {piece.gnd_ph_count.median():.0f}")
ph_tbl.T                                                                 # Число фотонов по классам.

**Вопрос:** Опишите облако фотонов: где лежат фотоны земли, полога и шума? Сколько фотонов земли приходится на сегмент? Почему высоту земли сегмента считают медианой, а не средним?

**Ответ:** _______________

### **6.3. Фильтр качества**

Сначала посмотрим, сколько сегментов оставляют разные пороги числа фотонов земли под высоким лесом и на открытой местности (готовая ячейка).

In [ ]:
q0 = isat[isat.h_ground.notna() & (isat.h_ground != 0)]                  # Сегменты, где найдена земля.
tall = q0.h_canopy.fillna(0) > 20                                        # Под высоким лесом.
open_ = q0.h_canopy.fillna(0) < 2                                        # На открытой местности.
rows = []                                                                # Строки таблицы.
for thr in (10, 20, 40):                                                 # Три порога:
    keep = q0.gnd_ph_count >= thr                                        # Сегменты, прошедшие порог.
    rows.append([thr, int(keep.sum()), 100 * keep.sum() / len(isat), 100 * (1 - keep[tall].mean()), 100 * (1 - keep[open_].mean())])
thr_tbl = pd.DataFrame(rows, columns=["Порог фотонов земли", "Остаётся сегментов", "Доля от исходных, %", "Отсеяно под лесом выше 20 м, %", "Отсеяно на открытой местности, %"]).round(1)
fig, ax = plt.subplots(figsize=(10, 4))                                  # Распределения числа фотонов земли.
bins = np.arange(0, 200, 5)                                              # Интервалы гистограммы.
ax.hist(q0.gnd_ph_count[open_].clip(0, 199), bins=bins, alpha=0.6, label="открытая местность", color="#e8d36b")
ax.hist(q0.gnd_ph_count[tall].clip(0, 199), bins=bins, alpha=0.6, label="лес выше 20 м", color="#1f7a3a")
for thr in (10, 20, 40): ax.axvline(thr, color="k", ls=":"); ax.text(thr + 1, ax.get_ylim()[1] * 0.9, str(thr), fontsize=9)
ax.set_xlabel("Фотонов земли в сегменте"); ax.set_ylabel("Сегментов"); ax.legend(); ax.set_title("Число фотонов земли: открытая местность и высокий лес")
plt.show()                                       # Показываем рисунок.
thr_tbl                                                                  # Показываем таблицу.

**Задание 6.3.** Выберите порог `MIN_PHOTONS` по таблице и рисунку выше и отфильтруйте сегменты: высота земли определена (не пропуск и не 0) и фотонов земли не меньше порога. Сохраните результат в `isat_q` и постройте таблицу «воронки» `funnel_df`: сколько сегментов остаётся после каждого шага и какая это доля от исходных.

In [ ]:
MIN_PHOTONS = ...                                                        # Ваш порог числа фотонов земли.
found = ...                                                              # Высота земли определена: h_ground не пропуск и не 0.
enough = ...                                                             # gnd_ph_count не меньше MIN_PHOTONS.
isat_q = ...                                                             # Сегменты, прошедшие оба фильтра.
funnel_df = ...                                                          # Таблица «воронки»: шаг, осталось сегментов, доля от исходных, %.
funnel_df

**Вопрос:** Какой порог вы выбрали и почему? Какую долю измерений под высоким лесом и на открытой местности он отсеивает и что произойдёт с эталоном в лесу, если порог сильно увеличить?

**Ответ:** _______________

### **6.4. Пересчёт высот и завышение по классам покрова**

In [ ]:
q = isat_q.copy()                                                        # Сегменты после фильтров качества.
q["h_egm08"] = ellipsoid_to_geoid(q.h_ground.values, q.lon.values, q.lat.values, "EGM2008")   # Высота над геоидом EGM2008.
q["h_egm96"] = ellipsoid_to_geoid(q.h_ground.values, q.lon.values, q.lat.values, "EGM96")     # То же над EGM96.
row, col = lonlat_to_rowcol(q.lon.values, q.lat.values, GRID)            # Положение точек на сетке участка.
inside = (row > 1) & (row < GRID["H"] - 2) & (col > 1) & (col < GRID["W"] - 2)   # Отбрасываем точки у самого края сетки.
q, row, col = q[inside].copy(), row[inside], col[inside]                 # Оставляем точки внутри.
q["row"], q["col"] = row, col                                            # Запоминаем положение на сетке.
q["cop"] = map_coordinates(stack["cop"], [row, col], order=1)            # Высота Copernicus в точке (билинейная интерполяция).
open_mask = q.h_canopy.fillna(0) < 2                                     # Открытая местность: растительность ниже 2 м.
datum_check = pd.DataFrame([                                             # Медианная разность без пересчёта и с ним.
    ["без пересчёта (эллипсоид)", np.median(q.cop - q.h_ground), np.median((q.cop - q.h_ground)[open_mask])],
    ["с пересчётом в EGM2008", np.median(q.cop - q.h_egm08), np.median((q.cop - q.h_egm08)[open_mask])],
], columns=["Высоты ICESat-2", "Медиана (Copernicus − ICESat-2), м", "То же на открытой местности, м"]).round(2)
datum_check                                                              # Показываем таблицу.

**Вопрос:** На сколько метров различаются лазерные высоты и модель рельефа без пересчёта систем высот и после него — в целом и на открытой местности? Откуда берётся этот сдвиг?

**Ответ:** _______________

In [ ]:
ri, ci = np.round(q.row).astype(int).values, np.round(q.col).astype(int).values   # Ячейка сетки для каждой точки.
q["wc"] = stack["wc"][ri, ci].astype(int)                                # Класс WorldCover в точке.
q["lossyear"] = stack["lossyear"][ri, ci]                                # Год потери леса (0 – потери не было).
q["dh"] = q.cop - q.h_egm08                                              # Завышение модели рельефа после пересчёта, м.
q["cls_wc"] = q.wc.map(WC_TO_CLASS).fillna(0).astype(int)                # Класс покрова работы по WorldCover.
rows = []                                                                # Строки таблицы.
for c in sorted(set(q.cls_wc) - {0}):                                    # Для каждого класса:
    m_ = (q.cls_wc == c).values
    if m_.sum() >= 50:
        rows.append([CLASS_NAMES[c], int(m_.sum()), np.median((q.cop - q.h_ground)[m_]), np.median(q.dh[m_])])
by_class = pd.DataFrame(rows, columns=["Класс покрова", "Сегментов", "Медиана Copernicus − ICESat-2 без пересчёта, м", "После пересчёта в EGM2008, м"]).round(2)
by_class                                                                 # Показываем таблицу.

**Вопрос:** Чему равно завышение модели рельефа в каждом классе покрова после пересчёта? Почему без пересчёта сдвиг почти одинаков для всех классов, а после пересчёта классы расходятся?

**Ответ:** _______________

### **6.5. Очистка измерений**

In [ ]:
tall = q.h_canopy.fillna(0) > 20                                         # Под высоким лесом.
open_ = q.h_canopy.fillna(0) < 2                                         # Открытая местность.
rows = []                                                                # Строки таблицы.
for thr in (50, 20, 10):                                                 # Три порога:
    out = q.dh.abs() > thr                                               # Сегменты, которые порог исключил бы.
    rows.append([thr, int(out.sum()), 100 * out[tall].mean(), 100 * out[open_].mean()])
outlier_tbl = pd.DataFrame(rows, columns=["Порог |Copernicus − ICESat-2|, м", "Исключено сегментов", "Доля исключённых под лесом выше 20 м, %", "Доля исключённых на открытой местности, %"]).round(1)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1, 1.3]})   # Гистограмма и карта.
ax[0].hist(q.dh.clip(-60, 60), bins=120, color="#2e86de"); ax[0].set_yscale("log"); ax[0].set_xlabel("Copernicus − ICESat-2, м"); ax[0].set_ylabel("Сегментов (лог. шкала)")
for thr in (10, 20, 50): ax[0].axvline(thr, color="k", ls=":"); ax[0].axvline(-thr, color="k", ls=":")
ax[0].set_title("Распределение разности высот и пороги")
ax[1].imshow(ls.hillshade(dem, vert_exag=3, dx=30, dy=30), cmap="gray", alpha=0.6)   # Подложка – отмывка рельефа.
big = q.dh.abs() > 50                                                    # Грубые выбросы.
ax[1].scatter(q.col[~big], q.row[~big], s=0.3, c="#bbbbbb"); ax[1].scatter(q.col[big], q.row[big], s=12, c="#c0392b", label=f"|разность| > 50 м: {int(big.sum())}")
ax[1].legend(loc="lower left"); ax[1].axis("off"); ax[1].set_title("Где лежат грубые выбросы")
plt.tight_layout(); plt.show()                         # Показываем рисунок.
outlier_tbl                                                              # Показываем таблицу.

**Вопрос:** Опишите гистограмму разности высот: сколько у неё «горбов» и чему они соответствуют? Сколько сегментов исключили бы пороги 50, 20 и 10 м и почему порог 10 м нельзя использовать, хотя гистограмма выглядела бы «чище»?

**Ответ:** _______________

**Задание 6.5.** Допишите маски четырёх правил очистки в вызовах `drop` (функция исключает сегменты и записывает шаг в журнал). Порог грубых выбросов `MAX_DH = 50` м задан.

In [ ]:
ri, ci = np.round(q.row).astype(int).values, np.round(q.col).astype(int).values   # Ячейка сетки для каждой точки.
q["wc"] = stack["wc"][ri, ci].astype(int)                                # Класс WorldCover в точке.
q["lossyear"] = stack["lossyear"][ri, ci]                                # Год потери леса (0 – потери не было).
q["dh"] = q.cop - q.h_egm08                                              # Завышение модели рельефа, м.
MAX_DH = 50                                                              # Порог грубых выбросов, м.
log = []                                                                 # Журнал очистки.

# Ваш код: замените ... масками правил (логические Series той же длины, что q):
#   снег – флаг snowcover больше 1; вода – класс 80 WorldCover (столбец wc) или классы 80, 200 ATL08 (столбец landcover);
#   изменения поверхности – lossyear не меньше 11; грубые выбросы – |dh| больше MAX_DH.
def drop(data, mask, what, rule, why):
    """Исключает строки по маске и записывает шаг в журнал очистки."""
    log.append([what, rule, int(mask.sum()), why])                       # Что исключено, правило, сколько, почему.
    return data[~mask]                                                   # Оставляем только пригодные измерения.

q = drop(q, ..., "Измерения со снегом", "флаг снега ICESat-2 > 1", "снег поднимает поверхность, сезон не совпадает с моделью рельефа")
q = drop(q, ..., "Водные объекты", "класс воды WorldCover или ATL08", "уровень воды меняется, в моделях рельефа водная поверхность выровнена")
q = drop(q, ..., "Вырубки и гари после 2010 г.", "год потери леса Hansen ≥ 2011", "поверхность изменилась после съёмки модели рельефа")
q = drop(q, ..., "Грубые выбросы", f"|Copernicus − ICESat-2| > {MAX_DH} м", "ошибочные отражения (облака, шум); физически неправдоподобно")
clean_log = pd.DataFrame(log, columns=["Что исключено", "Правило", "Затронуто сегментов", "Обоснование"])  # Журнал как таблица.
print(f"После очистки осталось {len(q)} сегментов (условие пригодности ≥ 3000: {'выполнено' if len(q) >= 3000 else 'НЕ выполнено'})")
clean_log                                                                # Показываем таблицу.

**Вопрос:** Какое правило исключило больше всего сегментов и почему на вашей территории? Сколько сегментов осталось и хватит ли их для обучения модели (не меньше 3000)?

**Ответ:** _______________

---

## **7. Классификация пикселей**

> 📚 **Подсказка:** лекция «Цифровая обработка спутниковых изображений рельефа и гравитационного поля», **раздел 6**; [лекция по GEE и машинному обучению](https://colab.research.google.com/drive/1BU-Q4D0XEIy2KiEtzw-3dKISE0lwGLV2), раздел 4 (классификаторы); [практическая работа №5](https://colab.research.google.com/drive/1mwCV-82k9R5ZtR0NGVJT3oLNyKDG0ZXO), раздел 4.2.

In [ ]:
label_map = np.zeros(stack["wc"].shape, int)                             # Метки покрова; 0 – класс не используется.
for wc_code, our_class in WC_TO_CLASS.items():                           # Для каждого кода WorldCover:
    label_map[stack["wc"] == wc_code] = our_class                        # Переводим коды WorldCover в классы работы.
counts = pd.Series(label_map.ravel()).value_counts()                     # Число ячеек каждого класса.
classes_used = [c for c in CLASS_NAMES if counts.get(c, 0) >= 200]       # Классы, которых на участке достаточно.
balance = pd.DataFrame({"Класс": [CLASS_NAMES[c] for c in classes_used], "Ячеек": [int(counts[c]) for c in classes_used]})  # Число ячеек по классам.
balance["Доля, %"] = (100 * balance["Ячеек"] / balance["Ячеек"].sum()).round(1)  # Доля класса на участке.
balance                                                                  # Показываем таблицу.

**Вопрос:** Какие классы покрова есть на вашей территории и какую долю площади занимает каждый? Почему из каждого класса в обучение берётся не более 4000 пикселей?

**Ответ:** _______________

### **7.1. Случайная и пространственная проверка**

**Задание 7.1.** В ячейке подготовлены признаки, выборка пикселей и функции оценки. Получите общую точность и средний F1 классификатора двумя способами: при случайном разбиении на пять частей (`KFold(5, shuffle=True, random_state=SEED)`) и при разбиении по блокам 2 км (`GroupKFold(5)` с группами `blk2[idx]`). Для модели высот это уже сделано в той же ячейке.

In [ ]:
S2B = ["B2", "B3", "B4", "B8", "B11", "B12", "NDVI", "NDWI", "NDBI"]    # Спектральные признаки.
RELB = ["slope", "tpi7", "tpi31", "rough7"]                              # Признаки рельефа (кроме высоты).
F_s2 = np.stack([stack[b] for b in S2B], -1).reshape(-1, len(S2B))       # Таблица «ячейка × спектральный признак».
F_rel = np.column_stack([dem.ravel()] + [REL[k].ravel() for k in RELB])  # Таблица «ячейка × признак рельефа».
X_b = np.hstack([F_s2, F_rel])                                           # Все признаки.
y_all = label_map.ravel()                                                # Метка каждой ячейки.
rr, cc = np.indices(dem.shape)                                           # Строка и столбец каждой ячейки.
blk2 = block_id(rr.ravel(), cc.ravel(), GRID, 2000)                      # Номер блока 2 км для каждой ячейки.
rng = np.random.default_rng(SEED)                                        # Генератор случайных чисел с зерном.
idx = np.concatenate([rng.choice(np.where(y_all == c)[0], min(int(counts[c]), 4000), replace=False) for c in classes_used])   # До 4000 ячеек на класс.
def rf():
    """Новый классификатор Random Forest."""
    return RandomForestClassifier(300, min_samples_leaf=2, n_jobs=N_JOBS, random_state=SEED)
def cv_accuracy(splits):
    """Общая точность и средний F1 по прогнозам вне обучения для заданного разбиения."""
    pred = np.zeros(len(idx), int)
    for tr, te in splits:                                                # Для каждой части:
        pred[te] = rf().fit(X_b[idx][tr], y_all[idx][tr]).predict(X_b[idx][te])   # обучаем на остальных, предсказываем её.
    return accuracy_score(y_all[idx], pred), f1_score(y_all[idx], pred, average="macro")
acc_rand = ...                                                           # Ваш код: cv_accuracy(разбиение KFold).
acc_blk = ...                                                            # Ваш код: cv_accuracy(разбиение GroupKFold по блокам blk2[idx]).
FEAT = {"wcl": label_map, **REL, **{b: stack[b] for b in ["B4", "B8", "B11", "NDVI", "NDWI"]}, "eth": stack["eth"]}   # Признаки модели высот.
ri, ci = np.round(q.row).astype(int).values, np.round(q.col).astype(int).values   # Ячейка сетки для каждой точки.
for k, a in FEAT.items():                                                # Значение каждого признака в точке.
    q[k] = a[ri, ci]
qh = q[q.wcl != 6].copy()                                                # Воду модель высот не уточняет.
F_H = ["wcl", "slope", "tpi7", "tpi31", "rough7", "B4", "B8", "B11", "NDVI", "NDWI", "eth"]   # Признаки модели высот.
yh = qh.dh.values; blk5 = block_id(qh.row.values, qh.col.values, GRID, 5000)   # Целевая переменная и блоки 5 км.
def hgb():
    """Новая модель градиентного бустинга для уточнения высот."""
    return HistGradientBoostingRegressor(loss="absolute_error", max_iter=400, learning_rate=0.05, categorical_features=[0], random_state=SEED)
def cv_rmse(splits):
    """RMSE по прогнозам вне обучения для заданного разбиения."""
    pred = np.zeros(len(qh))
    for tr, te in splits:
        pred[te] = hgb().fit(qh[F_H].iloc[tr], yh[tr]).predict(qh[F_H].iloc[te])
    return height_metrics(yh - pred)["RMSE"]
rmse_rand = cv_rmse(KFold(5, shuffle=True, random_state=SEED).split(qh))                   # Случайное разбиение.
rmse_blk = cv_rmse(GroupKFold(5).split(qh, yh, blk5))                                      # Разбиение по блокам 5 км.
leak = pd.DataFrame([["Классификация покрова: общая точность", acc_rand[0], acc_blk[0]], ["Классификация покрова: средний F1", acc_rand[1], acc_blk[1]],
                     ["Модель высот: RMSE, м", rmse_rand, rmse_blk]], columns=["Показатель", "Случайное разбиение", "По пространственным блокам"]).round(3)
fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))                          # Карты блоков.
for a, size, t in [(ax[0], 2000, "Блоки 2 км (классификация)"), (ax[1], 5000, "Блоки 5 км (модель высот)")]:
    a.imshow(ls.hillshade(dem, vert_exag=3, dx=30, dy=30), cmap="gray", alpha=0.6)
    a.imshow(block_id(rr, cc, GRID, size) % 7, cmap="tab10", alpha=0.35, interpolation="nearest"); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()                           # Показываем рисунок.
leak                                                                     # Показываем таблицу.

**Вопрос:** На сколько точность классификатора и ошибка модели высот при случайном разбиении лучше, чем при разбиении по блокам? Почему случайная проверка завышает точность для пикселей изображения и какую оценку нужно приводить?

**Ответ:** _______________

### **7.2. Снимок против снимка с рельефом**

In [ ]:
S2B = ["B2", "B3", "B4", "B8", "B11", "B12", "NDVI", "NDWI", "NDBI"]    # Спектральные признаки.
RELB = ["slope", "tpi7", "tpi31", "rough7"]                              # Признаки рельефа (кроме высоты).
F_s2 = np.stack([stack[b] for b in S2B], -1).reshape(-1, len(S2B))       # Таблица «ячейка × спектральный признак».
F_rel = np.column_stack([dem.ravel()] + [REL[k].ravel() for k in RELB])  # Таблица «ячейка × признак рельефа».
y_all = label_map.ravel()                                                # Метка каждой ячейки.
rr, cc = np.indices(dem.shape)                                           # Строка и столбец каждой ячейки.
blk2 = block_id(rr.ravel(), cc.ravel(), GRID, 2000)                      # Номер блока 2 км для каждой ячейки.
rng = np.random.default_rng(SEED)                                        # Генератор случайных чисел с зерном.
parts = []                                                               # Номера ячеек, выбранных по классам.
for c in classes_used:                                                   # Для каждого класса:
    cells_c = np.where(y_all == c)[0]                                    # Все ячейки класса c.
    n_take = min(len(cells_c), 4000)                                     # Не более 4000 ячеек на класс.
    parts.append(rng.choice(cells_c, n_take, replace=False))             # Случайная выборка без повторов.
idx = np.concatenate(parts)                                              # Номера ячеек обучающей выборки.

def cv_classify(X, model_fn):
    """Предсказания вне обучения по пространственным блокам."""
    pred = np.zeros(len(idx), int)
    for tr, te in GroupKFold(5).split(X[idx], y_all[idx], blk2[idx]):    # Пять частей, блоки не разрываются.
        model = model_fn().fit(X[idx][tr], y_all[idx][tr])               # Обучаем на четырёх частях,
        pred[te] = model.predict(X[idx][te])                             # предсказываем пятую.
    return pred

def rf():
    """Новый классификатор Random Forest."""
    return RandomForestClassifier(300, min_samples_leaf=2, n_jobs=N_JOBS, random_state=SEED)   # 300 деревьев, в листе ≥ 2 ячеек.

X_a = F_s2                                                               # Признаки: только снимок.
X_b = np.hstack([F_s2, F_rel])                                           # Признаки: снимок + рельеф.
pred_a, pred_b = cv_classify(X_a, rf), cv_classify(X_b, rf)              # Прогнозы вне обучения для обоих наборов.
rows = []                                                                # Строки будущей таблицы.
for name, p_ in [("только снимок", pred_a), ("снимок + рельеф", pred_b)]:  # Для каждого набора признаков:
    rows.append([name, accuracy_score(y_all[idx], p_), cohen_kappa_score(y_all[idx], p_), f1_score(y_all[idx], p_, average="macro")])  # Три показателя качества.
q1 = pd.DataFrame(rows, columns=["Признаки", "Общая точность", "Каппа Коэна", "Средний F1 по классам"]).round(3)  # Таблица сравнения наборов признаков.
q1                                                                       # Показываем таблицу.

**Вопрос:** Насколько признаки рельефа повысили общую точность, каппу и средний F1 на вашей территории? Почему высота и уклон помогают различать классы, похожие по спектру?

**Ответ:** _______________

### **7.3. Сравнение классификаторов**

**Задание 7.3.** Сравните на признаках «снимок + рельеф» (`X_b`) все классификаторы, которые разбирались в [лекции по GEE](https://colab.research.google.com/drive/1BU-Q4D0XEIy2KiEtzw-3dKISE0lwGLV2), — их аналоги в scikit-learn:

| Классификатор в GEE | Аналог в scikit-learn |
|---|---|
| `smileRandomForest` | `RandomForestClassifier(300, min_samples_leaf=2)` |
| `smileCart` | `DecisionTreeClassifier(min_samples_leaf=5)` |
| `smileGradientTreeBoost` | `HistGradientBoostingClassifier(max_iter=300)` |
| `libsvm` (ядро RBF) | `make_pipeline(StandardScaler(), SVC(C=10))` |
| `smileNaiveBayes` | `GaussianNB()` |
| `minimumDistance` | `make_pipeline(StandardScaler(), NearestCentroid())` |

Для каждого получите прогноз вне обучения по блокам 2 км (`GroupKFold(5)`, группы `blk2[idx]`; на каждой части — новая копия модели `clone(model)`), сохраните прогнозы в словарь `preds`, а в таблицу `cls_cmp` — общую точность, каппу, средний F1 и время. Название лучшего по каппе запишите в `best_cls`. SVM обучается дольше остальных (около минуты).

In [ ]:
from sklearn.base import clone                                           # Новая копия модели с теми же параметрами.
from sklearn.tree import DecisionTreeClassifier                          # CART – дерево решений.
from sklearn.naive_bayes import GaussianNB                               # Наивный байесовский классификатор.
from sklearn.neighbors import NearestCentroid                            # Классификатор минимального расстояния.

models_cls = {                                                           # Классификаторы из лекции по GEE.
    "Random Forest": RandomForestClassifier(300, min_samples_leaf=2, n_jobs=N_JOBS, random_state=SEED),
    # Ваш код: CART, Gradient Boost, SVM (RBF), Naive Bayes, Min Distance
    # (SVM и Min Distance – в цепочке make_pipeline(StandardScaler(), ...))
}
rows, preds = [], {}                                                     # Строки таблицы и прогнозы вне обучения.
# Ваш код: для каждого классификатора – прогноз вне обучения по GroupKFold(5) с группами blk2[idx]
#          (на каждой части – clone(model).fit(...).predict(...)), запись в preds[name],
#          строка таблицы: общая точность, каппа, средний F1, время
cls_cmp = ...                                                            # Таблица, отсортированная по каппе.
best_cls = ...                                                           # Название лучшего классификатора.
cls_cmp

**Вопрос:** Какой классификатор оказался лучшим и насколько он опережает остальных? Какие классификаторы заметно уступают и почему (вспомните, какие предположения о данных делают наивный Байес и классификатор минимального расстояния)? Стоит ли выигрыш в точности затраченного времени?

**Ответ:** _______________

### **7.4. Матрица ошибок и показатели по классам**

**Задание 7.4.** Для лучшего классификатора постройте матрицу ошибок, нормированную по строкам (по эталону WorldCover), и таблицу `per_class`: точность производителя (`recall_score`), точность пользователя (`precision_score`) и F1 для каждого класса (`average=None`, `labels=labels_present`).

In [ ]:
y_true, y_pred = y_all[idx], preds[best_cls]                             # Эталон WorldCover и прогноз вне обучения.
labels_present = sorted(set(y_true))                                     # Классы выборки.
names = [CLASS_NAMES[c] for c in labels_present]
cm = ...                                                                 # confusion_matrix(..., labels=labels_present, normalize="true")
# Ваш код: рисунок матрицы ошибок (ConfusionMatrixDisplay) с названиями классов
per_class = ...                                                          # Таблица: класс, точность производителя (recall), точность пользователя (precision), F1
per_class

**Вопрос:** Какой класс распознаётся лучше всего, а какой хуже всего? С каким классом путается худший класс (по строке матрицы) и почему они похожи по спектру и рельефу? Для какого класса точность пользователя заметно ниже точности производителя и что это значит для карты?

**Ответ:** _______________

### **7.5. Карта покрова**

**Задание 7.5.** Обучите лучший классификатор (`clone(models_cls[best_cls])`) на всей выборке `X_b[idx]`, `y_all[idx]` и классифицируйте все пиксели изображения `X_b` частями по 200 000 пикселей. Результат — карта `cls` размером `dem.shape`; она нужна во всех следующих разделах.

In [ ]:
clf = ...                                                                # Лучший классификатор, обученный на всей выборке X_b[idx], y_all[idx].
cls = ...                                                                # Прогноз для всех пикселей X_b частями по 200 000, затем .reshape(dem.shape).
print("Классы на карте:", sorted(set(cls.ravel())))

In [ ]:
cmap_cls = ListedColormap([CLASS_COLORS[i] for i in range(1, 8)])        # Палитра из цветов классов 1–7.
norm_cls = BoundaryNorm(np.arange(0.5, 8.5), 7)                          # Каждому номеру класса – свой цвет.
fig, ax = plt.subplots(1, 2, figsize=(13, 6))                            # Два рисунка рядом.
ax[0].imshow(rgb); ax[0].set_title("Снимок")                             # Слева – снимок.
ax[1].imshow(cls, cmap=cmap_cls, norm=norm_cls, interpolation="nearest"); ax[1].set_title("Карта покрова (модель ИИ)")  # Справа – карта покрова.
legend = [Patch(color=CLASS_COLORS[c], label=CLASS_NAMES[c]) for c in classes_used]   # Элементы легенды.
ax[1].legend(handles=legend, loc="lower left", fontsize=9)               # Легенда классов.
for a in ax: a.axis("off")                                               # Без осей координат.
plt.tight_layout(); plt.show()                   # Показываем рисунок.
areas = pd.DataFrame({"Класс": [CLASS_NAMES[c] for c in classes_used],   # Площадь каждого класса на карте:
                      "Площадь, км²": [round((cls == c).sum() * 900 / 1e6, 1) for c in classes_used]})   # Ячейка – 900 м².
areas                                                                    # Показываем таблицу.

In [ ]:
mask_train = np.zeros(label_map.size, bool); mask_train[idx] = True   # Ячейки обучающей выборки.
err = (cls != label_map) & (label_map > 0) & ~mask_train.reshape(dem.shape)   # Расхождение с WorldCover вне обучения.
fig, ax = plt.subplots(figsize=(8, 7))                                   # Карта расхождений.
ax.imshow(rgb); ax.imshow(np.where(err, 1.0, np.nan), cmap="autumn", vmin=0, vmax=1, alpha=0.7)   # Красным – расхождения.
ax.set_title(f"Расхождение карты покрова с WorldCover: {100 * err.sum() / ((label_map > 0) & ~mask_train.reshape(dem.shape)).sum():.1f}% ячеек вне обучения"); ax.axis("off")
plt.show()                                        # Показываем рисунок.

**Вопрос:** Где карта покрова расходится с WorldCover — на границах классов, в смешанных участках, в каких-то типах местности? Приведите долю расхождений. Можно ли по ним судить, что ошибается именно ваша модель, а не WorldCover?

**Ответ:** _______________

---

## **8. Уточнение модели рельефа**

> 📚 **Подсказка:** лекция «Цифровая обработка спутниковых изображений рельефа и гравитационного поля», **раздел 7**.

### **8.1. Показатели точности высот**

**Задание 8.1.** Напишите функцию `my_height_metrics(e)`, которая возвращает словарь показателей ошибки высот: RMSE, MAE, медианное смещение, NMAD и LE90 (формулы — в таблице раздела 7.1 лекции). Проверьте её на разности `q.dh`: значения должны совпасть с готовой функцией `height_metrics`.

In [ ]:
def my_height_metrics(e):
    """RMSE, MAE, медианное смещение, NMAD и LE90 для ошибок высот e."""
    e = np.asarray(e)
    return {"RMSE": ..., "MAE": ..., "смещение": ..., "NMAD": ..., "LE90": ...}   # Формулы – в таблице показателей лекции (раздел 7.1).

pd.DataFrame([my_height_metrics(q.dh), height_metrics(q.dh)], index=["своя функция", "готовая функция"]).round(3)

**Вопрос:** Почему RMSE исходной модели заметно больше NMAD? Что это говорит о распределении ошибок (вспомните двугорбую гистограмму из раздела 6.5)?

**Ответ:** _______________

### **8.2. Признаки в точках и простые ориентиры**

Готовая ячейка добавляет к каждой точке признаки (класс покрова по вашей карте и по WorldCover, рельеф, снимок, высоту леса ETH), задаёт блоки 5 км и рассчитывает прогнозы простых правил: без поправки, общий сдвиг, медиана завышения по классу покрова.

In [ ]:
ri, ci = np.round(q.row).astype(int).values, np.round(q.col).astype(int).values   # Ячейка сетки для каждой точки.
FEAT = {"cls": cls, "wcl": label_map}                                    # Класс покрова: своя карта и WorldCover.
FEAT.update(REL)                                                         # Уклон, TPI, шероховатость.
for b in ["B4", "B8", "B11", "NDVI", "NDWI"]:                            # Для каждого спектрального признака:
    FEAT[b] = stack[b]                                                   # Каналы и индексы Sentinel-2.
FEAT["eth"] = stack["eth"]                                               # Высота леса ETH.
for k, a in FEAT.items():                                                # Для каждого признака:
    q[k] = a[ri, ci]                                                     # Значение признака в ячейке каждой точки.
q = q[q.cls != 6].copy()                                                 # Воду модель высот не уточняет.
y = q.dh.values                                                          # Целевая переменная: завышение, м.
blk5 = block_id(q.row.values, q.col.values, GRID, 5000)                  # Номер блока 5 км для каждой точки.
F_BASE = ["cls", "slope", "tpi7", "tpi31", "rough7", "B4", "B8", "B11", "NDVI", "NDWI"]   # Признаки без высоты леса.
F_ETH = F_BASE + ["eth"]                                                 # Признаки с высотой леса.

def hgb():
    """Новая модель градиентного бустинга для уточнения высот."""
    return HistGradientBoostingRegressor(loss="absolute_error", max_iter=400, learning_rate=0.05,   # 400 деревьев, шаг 0,05.
                                         categorical_features=[0], random_state=SEED)              # Класс покрова – категория.

def cv_correction(df, y, groups, feats):
    """Прогноз завышения вне обучения по пространственным блокам 5 км."""
    pred = np.zeros(len(df))
    for tr, te in GroupKFold(5).split(df, y, groups):
        model = hgb().fit(df[feats].iloc[tr], y[tr])                     # Обучаем на четырёх частях,
        pred[te] = model.predict(df[feats].iloc[te])                     # предсказываем пятую.
    return pred

n = len(q)                                                               # Число точек.
P = {"Без поправки": np.zeros(n), "Общий сдвиг": np.zeros(n), "По классам WorldCover": np.zeros(n), "По своей карте покрова": np.zeros(n)}  # Прогнозы поправки каждого метода.
for tr, te in GroupKFold(5).split(q, y, blk5):                           # Простые правила на тех же частях.
    P["Общий сдвиг"][te] = np.median(y[tr])                              # Медиана завышения по обучающим блокам.
    for key, c in [("По классам WorldCover", "wcl"), ("По своей карте покрова", "cls")]:  # Две карты покрова:
        med = pd.Series(y[tr]).groupby(q[c].values[tr]).median()         # Медиана завышения в каждом классе.
        by_class = pd.Series(q[c].values[te]).map(med)                   # Поправка по классу точки,
        P[key][te] = by_class.fillna(np.median(y[tr])).values            # для класса без обучающих точек – общий сдвиг.
forest = (q.cls == 1).values                                             # Точки в лесу.

def r_squared(e):
    """Доля разброса завышения, объяснённая поправкой (R²)."""
    return 1 - np.sum(e ** 2) / np.sum((y - y.mean()) ** 2)
print(f"Сегментов ICESat-2: {len(q)}, пространственных блоков 5 км: {len(set(blk5))}")

### **8.3. Сравнение моделей уточнения высот**

**Задание 8.3.** Сравните четыре модели регрессии, предсказывающие завышение `y` по признакам `F_ETH`: градиентный бустинг (задан), случайный лес (`RandomForestRegressor(300, min_samples_leaf=5, max_features=0.5)`), дерево решений (`DecisionTreeRegressor(min_samples_leaf=20)`) и регрессию по k ближайшим соседям (`make_pipeline(StandardScaler(), KNeighborsRegressor(15))`). Для каждой получите прогноз вне обучения по блокам 5 км и запишите его в словарь `P` рядом с простыми правилами. Постройте таблицу `res` для всех методов из `P`: показатели `height_metrics`, R² (`r_squared`) и медианное смещение в лесу (`forest`). Название лучшей модели по RMSE запишите в `best_reg` — дальше уточнённый рельеф строится ею.

In [ ]:
from sklearn.ensemble import RandomForestRegressor                      # Случайный лес для регрессии.
from sklearn.tree import DecisionTreeRegressor                          # Дерево решений для регрессии.
from sklearn.neighbors import KNeighborsRegressor                        # Регрессия по k ближайшим соседям.

reg_models = {                                                           # Модели уточнения высот.
    "Градиентный бустинг": HistGradientBoostingRegressor(loss="absolute_error", max_iter=400, learning_rate=0.05,
                                                         categorical_features=[0], random_state=SEED),
    # Ваш код: случайный лес, дерево решений, k ближайших соседей (в цепочке со StandardScaler)
}
# Ваш код: прогноз вне обучения каждой моделью по GroupKFold(5) с группами blk5 на признаках F_ETH, запись в P[name]
# Ваш код: таблица res – для каждого метода из P показатели height_metrics(y − прогноз), R² (r_squared) и смещение в лесу (forest)
best_reg = ...                                                           # Модель с наименьшим RMSE среди reg_models.
res

**Вопрос:** Чему равны RMSE и смещение в лесу без поправки, у лучшего простого правила и у лучшей модели? Во сколько раз уточнение уменьшило ошибку? Какая модель оказалась лучшей и почему дерево решений и k ближайших соседей уступают ансамблям?

**Ответ:** _______________

In [ ]:
e_after = y - P[best_reg]                               # Ошибка после уточнения.
rows = []                                                                # Строки будущей таблицы.
for c in classes_used:                                                   # Для каждого класса:
    m_ = (q.cls == c).values                                             # Точки класса c.
    if m_.sum() >= 30:                                                   # Класс с малым числом точек не оцениваем.
        rows.append({"Класс": CLASS_NAMES[c], "Сегментов": int(m_.sum()),
                     "RMSE до, м": height_metrics(y[m_])["RMSE"], "RMSE после, м": height_metrics(e_after[m_])["RMSE"],
                     "Смещение до, м": np.median(y[m_]), "Смещение после, м": np.median(e_after[m_])})
per_cls = pd.DataFrame(rows).round(2)                                    # Таблица по классам.
per_cls                                                                  # Показываем таблицу.

**Вопрос:** В каком классе покрова уточнение уменьшило ошибку сильнее всего, а где почти ничего не изменило? Объясните почему.

**Ответ:** _______________

In [ ]:
q["oof"] = P[best_reg]                                 # Прогноз вне обучения в каждой точке.
e_oof = y - q.oof.values                                                 # Остаток после уточнения.
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))                          # Три диаграммы.
bins_eth = np.arange(0, 35, 2.5)                                         # Интервалы высоты леса.
for c, color, name in [(1, "#1f7a3a", "лес"), (3, "#b8a000", "луга и пашня"), (4, "#c0392b", "застройка")]:   # Три класса.
    m_ = (q.cls == c).values
    if m_.sum() < 100: continue
    g = pd.DataFrame({"eth": q.eth.values[m_], "dh": y[m_], "pred": q.oof.values[m_]}); g["bin"] = pd.cut(g.eth, bins_eth)
    agg = g.groupby("bin", observed=True).agg(dh=("dh", "median"), pred=("pred", "median"), n=("dh", "size")); agg = agg[agg.n >= 30]
    centers = [b.mid for b in agg.index]
    ax[0].plot(centers, agg.dh, "o-", color=color, label=f"{name}: измерено"); ax[0].plot(centers, agg.pred, "s--", color=color, alpha=0.6, label=f"{name}: прогноз")
ax[0].set_xlabel("Высота леса ETH, м"); ax[0].set_ylabel("Завышение модели рельефа, м (медиана)"); ax[0].set_title("Поправка растёт с высотой леса"); ax[0].legend(fontsize=8)
for a, feat, t in [(ax[1], q.slope.values, "Уклон, °"), (ax[2], q.cop.values, "Высота, м")]:   # Остатки по уклону и высоте.
    bins = np.quantile(feat, np.linspace(0, 1, 11)); mids = (bins[:-1] + bins[1:]) / 2
    lab = np.clip(np.digitize(feat, bins[1:-1]), 0, 9)
    a.plot(mids, [np.median(y[lab == b]) for b in range(10)], "o-", color="grey", label="до уточнения")
    a.plot(mids, [np.median(e_oof[lab == b]) for b in range(10)], "o-", color="#d35400", label="после уточнения")
    a.axhline(0, color="k", lw=0.8); a.set_xlabel(t); a.set_ylabel("Медиана ошибки, м"); a.legend(fontsize=8)
ax[1].set_title("Ошибка по уклону"); ax[2].set_title("Ошибка по высоте")
plt.tight_layout(); plt.show()              # Показываем рисунок.

**Вопрос:** Как завышение модели рельефа зависит от высоты леса в разных классах и повторяет ли прогноз эту зависимость? Как ведут себя ошибки до и после уточнения в зависимости от уклона и высоты?

**Ответ:** _______________

In [ ]:
rows = []                                                                # Строки таблицы.
rng = np.random.default_rng(SEED)                                        # Генератор случайных чисел.
for n_train in [500, 1000, 2000, 4000, 8000, len(q)]:                    # Объёмы обучения.
    errs = []                                                            # Ошибки по пяти частям.
    for tr, te in GroupKFold(5).split(q, y, blk5):                       # Проверка по блокам 5 км.
        sub = rng.choice(tr, min(n_train, len(tr)), replace=False)       # Подвыборка обучающих точек.
        pred = clone(reg_models[best_reg]).fit(q[F_ETH].iloc[sub], y[sub]).predict(q[F_ETH].iloc[te])   # Обучение на подвыборке, прогноз на части.
        errs.append(y[te] - pred)
    rows.append([min(n_train, len(q)), height_metrics(np.concatenate(errs))["RMSE"]])
curve = pd.DataFrame(rows, columns=["Обучающих сегментов", "RMSE на отложенных блоках, м"]).round(2)
fig, ax = plt.subplots(figsize=(7, 4)); ax.plot(curve.iloc[:, 0], curve.iloc[:, 1], "o-"); ax.set_xscale("log")
ax.set_xlabel("Обучающих сегментов (лог. шкала)"); ax.set_ylabel("RMSE, м"); ax.set_title("Кривая обучения модели высот"); ax.grid(alpha=0.3)
plt.show()                                     # Показываем рисунок.
curve                                                                    # Показываем таблицу.

**Вопрос:** Сколько очищенных лазерных измерений у вашей зоны и вышла ли кривая обучения на полку? Хватило бы вдвое меньшего числа измерений?

**Ответ:** _______________

### **8.4. Карта поправок и уточнённый рельеф**

In [ ]:
model_h = clone(reg_models[best_reg]).fit(q[F_ETH], y)                   # Итоговая модель: лучшая, обучаем на всех точках.
G = np.column_stack([FEAT[k].ravel() for k in F_ETH])                    # Признаки всех ячеек: строка – ячейка.
corr = model_h.predict(pd.DataFrame(G, columns=F_ETH)).reshape(dem.shape)   # Прогноз завышения, преобразованный обратно в карту.
corr[cls == 6] = 0                                                       # Воду не уточняем.
dtm = dem - corr                                                         # Уточнённая модель рельефа.
q["oof"] = P[best_reg]                                 # Прогноз вне обучения в каждой точке.
rows = []                                                                # Строки будущей таблицы.
for c in classes_used:                                                   # Для каждого класса:
    m_ = cls == c                                                        # Ячейки класса на карте покрова.
    if m_.sum() > 0:
        rows.append({"Класс": CLASS_NAMES[c], "Ячеек": int(m_.sum()),
                     "Поправка: медиана, м": np.median(corr[m_]), "Поправка: 95-й процентиль, м": np.percentile(corr[m_], 95)})
by_cls = pd.DataFrame(rows).round(2)                                     # Поправка по классам.
fig, ax = plt.subplots(1, 2, figsize=(13, 6))                            # Два рисунка рядом.
im = ax[0].imshow(corr, cmap="RdYlGn_r", vmin=-2, vmax=np.percentile(corr, 99)); ax[0].set_title("Поправка высот, м")  # Слева – карта поправки.
plt.colorbar(im, ax=ax[0], fraction=0.046)                               # Шкала значений.
ax[1].imshow(ls.hillshade(dtm, vert_exag=3, dx=30, dy=30), cmap="gray"); ax[1].set_title("Уточнённая модель рельефа")  # Справа – отмывка уточнённой модели.
for a in ax: a.axis("off")                                               # Без осей координат.
plt.tight_layout(); plt.show()                  # Показываем рисунок.
by_cls                                                                   # Показываем таблицу.

**Вопрос:** Какова медианная поправка высот в лесу и в других классах? Где на карте поправки самые большие и совпадает ли это с вашей гипотезой из раздела 3.2?

**Ответ:** _______________

In [ ]:
q["trk"] = q.rgt.astype(str) + "_" + q.cycle.astype(str) + "_" + q.spot.astype(str)   # Одна линия: трек, цикл, луч.
q["is_forest"] = q.cls == 1                                              # Точка в лесу.
tracks = q.groupby("trk").agg(n=("dh", "size"), forest=("is_forest", "mean"))   # Длина линии и доля леса на ней.
tracks = tracks[tracks.n >= 120]                                         # Только длинные линии.
score = tracks.n * tracks.forest * (1.2 - tracks.forest)                 # Длинная линия с долей леса около 60 %.
line = q[q.trk == score.idxmax()].sort_values("lat")                     # Точки выбранной линии с юга на север.
X_, Y_ = Transformer.from_crs("EPSG:4326", GRID["crs"], always_xy=True).transform(line.lon.values, line.lat.values)   # Координаты точек в метрах.
dist = np.hypot(X_ - X_[0], Y_ - Y_[0]) / 1000                           # Расстояние от начала линии, км.
plt.figure(figsize=(13, 4))                                              # Широкий рисунок для профиля.
plt.plot(dist, line.cop, color="grey", label="Copernicus DEM (исходная)")  # Исходная модель вдоль линии.
plt.plot(dist, line.cop - line.oof, color="#d35400", label="Уточнённая модель (прогноз вне обучения)")
plt.scatter(dist, line.h_egm08, s=6, color="k", label="ICESat-2 (эталон)", zorder=5)  # Лазерные высоты.
plt.xlabel("км вдоль трека"); plt.ylabel("высота, м (EGM2008)"); plt.legend()  # Подписи осей, легенда.
rmse_before, rmse_after = height_metrics(line.dh)["RMSE"], height_metrics(line.dh - line.oof)["RMSE"]  # Ошибка на линии до и после.
plt.title(f"RMSE вдоль трека: {rmse_before:.2f} → {rmse_after:.2f} м"); plt.show()  # Заголовок и показ рисунка.

**Вопрос:** Как изменилась ошибка вдоль лазерного трека? На каких участках профиля исходная модель расходится с эталоном и как ведёт себя уточнённая?

**Ответ:** _______________

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 6))                            # Два рисунка рядом.
for a, e, t in [(ax[0], q.dh, "Ошибка исходной модели, м"), (ax[1], q.dh - q.oof, "Ошибка уточнённой модели, м")]:  # Две карты ошибок:
    a.imshow(ls.hillshade(dem, vert_exag=3, dx=30, dy=30), cmap="gray", alpha=0.6)   # Подложка – отмывка рельефа.
    sc = a.scatter(q.col, q.row, c=e, s=2, cmap="RdBu_r", vmin=-10, vmax=10)          # Точки, окрашенные по ошибке.
    a.set_title(t); a.axis("off")                                        # Заголовок, без осей.
plt.colorbar(sc, ax=ax, fraction=0.03, label="Copernicus − ICESat-2, м"); plt.show()  # Общая шкала и показ рисунка.

**Вопрос:** Как изменилась карта остатков после уточнения? Остались ли участки, где ошибки сгруппированы, и с чем они могут быть связаны?

**Ответ:** _______________

In [ ]:
k_win = int(1200 / GRID["res"])                                          # Фрагмент 1,2×1,2 км на границе леса.
best, win = -np.inf, None
for r_ in range(0, GRID["H"] - k_win, k_win // 4):                               # Перебираем фрагменты с шагом 300 м:
    for c_ in range(0, GRID["W"] - k_win, k_win // 4):
        w_ = np.s_[r_:r_ + k_win, c_:c_ + k_win]
        fs = (cls[w_] == 1).mean()                                       # Доля леса во фрагменте.
        if not 0.4 <= fs <= 0.7 or (cls[w_] == 6).any():                 # Нужны и лес, и открытые участки, без воды.
            continue
        score = (corr[w_][cls[w_] == 1].mean() - corr[w_][cls[w_] != 1].mean()) / (np.ptp(dem[w_]) + 10)
        if score > best:                                                 # Большая разница поправки при небольшом перепаде высот.
            best, win = score, w_
Xs = np.arange(k_win) * GRID["res"] / 1000                               # Координаты узлов, км.
XX, YY = np.meshgrid(Xs, Xs[::-1])
fig = plt.figure(figsize=(17, 7))
ax = fig.add_subplot(1, 2, 1, projection="3d")                           # Две модели на одном рисунке.
ax.plot_surface(XX, YY, dtm[win], facecolors=rgb[win], rstride=1, cstride=1, linewidth=0, antialiased=False, shade=False)
ax.plot_wireframe(XX, YY, dem[win], color="#d62728", linewidth=0.9, rstride=2, cstride=2)   # Исходная модель – красная сетка.
ax.set_box_aspect((1, 1, 5 * np.ptp(dem[win]) / (k_win * GRID["res"])))  # Вертикальный масштаб ×5.
ax.view_init(elev=25, azim=-115); ax.zaxis.set_major_locator(plt.MaxNLocator(3))
ax.set_xlabel("км"); ax.set_ylabel("км"); ax.set_zlabel("Высота, м")
ax.set_title("Уточнённый рельеф (цвет – снимок) и исходная модель Copernicus (красная сетка), ×5")
ax = fig.add_subplot(1, 2, 2, projection="3d")
surface_3d(ax, corr[win], cmap="RdYlGn_r", z_ratio=0.35, title="Поправка высот (исходная − уточнённая), м", zlabel="Поправка, м")
plt.tight_layout(); plt.show()
print(f"Фрагмент: перепад высот {np.ptp(dem[win]):.0f} м, доля леса {100 * (cls[win] == 1).mean():.0f}%, средняя поправка в лесу {corr[win][cls[win] == 1].mean():.1f} м, вне леса {corr[win][cls[win] != 1].mean():.1f} м")

In [ ]:
rgb8 = (rgb * 255).astype(np.uint8)                                     # Цвета снимка в диапазоне 0–255.
cls_rgb = np.zeros(cls.shape + (3,), np.uint8)                           # Цветное изображение карты покрова.
for c, color in CLASS_COLORS.items():                                    # Для каждого класса:
    cls_rgb[cls == c] = np.round(np.array(to_rgb(color)) * 255)          # цвет класса из "#rrggbb" в три числа 0–255.
title = f"{TERRITORY}: трёхмерная модель местности (вертикальный масштаб ×3)"  # Заголовок модели.
fig3d = mesh3d_pair(dtm, rgb8, cls_rgb, classes_used, GRID, step=4, title=title)  # Две сцены, вершины через 120 м.
fig3d.show(post_script=SYNC_JS)                                          # Интерактивная модель: сцены поворачиваются вместе.

**Вопрос:** Как различаются исходная и уточнённая поверхности на трёхмерном фрагменте и где поверхность поправки выше всего? Что видно на интерактивной трёхмерной модели при сравнении снимка и карты покрова?

**Ответ:** _______________

---

## **9. Аномалии Буге**

> 📚 **Подсказка:** лекция «Цифровая обработка спутниковых изображений рельефа и гравитационного поля», **раздел 8** — что такое аномалия Буге и зачем она нужна, формула слоя Буге, цепочка фильтров.

In [ ]:
t = time.time()                                                          # Засекаем время.
LON, LAT = grid_lonlat(GRID)                                             # Долгота и широта центра каждой ячейки.
GGM_BOUNDS = [LON.min() - 0.01, LAT.min() - 0.01, LON.max() + 0.01, LAT.max() + 0.01]   # Сетка UTM по углам выходит за
dg_ll, g_lons, g_lats = load_ggmplus(GGM_BOUNDS, "dg")                  # границы, поэтому загружаем с запасом 0,01°.
zeta_ll, _, _ = load_ggmplus(GGM_BOUNDS, "geoid")                        # Высоты квазигеоида на той же сетке GGMplus.
fi, fj = (LAT - g_lats[0]) / 0.002, (LON - g_lons[0]) / 0.002            # Дробные номера узлов GGMplus для наших ячеек.
dg = map_coordinates(dg_ll, [fi, fj], order=1, mode="nearest")           # Возмущения силы тяжести на сетке участка, мГал.
zeta = map_coordinates(zeta_ll, [fi, fj], order=1, mode="nearest")       # Высоты квазигеоида на сетке участка, м.
assert not np.isnan(dg).any(), "На участке нет данных GGMplus (море или выход за 60° с. ш.)"  # Проверка: поле есть во всех ячейках.
print(f"Возмущения силы тяжести: {np.nanmin(dg):.1f} … {np.nanmax(dg):.1f} мГал; квазигеоид {np.nanmin(zeta):.2f} … {np.nanmax(zeta):.2f} м; {time.time() - t:.0f} с")
relief_smooth = gaussian_filter(stack["cop"], 8)                         # Сглаженный рельеф (масштаб поля GGMplus).
corr_relief = np.corrcoef(relief_smooth.ravel(), dg.ravel())[0, 1]       # Связь возмущений с рельефом.
print(f"Корреляция возмущений с рельефом: {corr_relief:.2f}")
ggm_params = pd.DataFrame([                                              # Параметры загрузки.
    ["Источник поля", "фрагмент вокруг участка из подготовленных данных" if ggm_source()[0] == "подготовленные данные" else f"{files_ru(len(ggm_source()[1]))} 5°×5° с сервера Curtin University (возмущения и квазигеоид)"],
    ["Границы загрузки", f"долгота {GGM_BOUNDS[0]:.3f}…{GGM_BOUNDS[2]:.3f}°, широта {GGM_BOUNDS[1]:.3f}…{GGM_BOUNDS[3]:.3f}°"],
    ["Шаг сетки GGMplus", "0,002° (около 220 м)"],
    ["Пересчёт на сетку участка", f"билинейная интерполяция, 30 м, {GRID['crs']}"],
    ["Возмущения силы тяжести, мГал", f"{np.nanmin(dg):.1f} … {np.nanmax(dg):.1f}"],
    ["Высота квазигеоида, м", f"{np.nanmin(zeta):.2f} … {np.nanmax(zeta):.2f}"],
    ["Корреляция возмущений с рельефом", f"{corr_relief:.2f}"],
], columns=["Параметр", "Значение"])
ggm_params                                                               # Показываем таблицу.

### **9.1. Цепочка расчёта своими руками**

**Задание 9.1.** Допишите цепочку расчёта аномалий Буге:
1. `smooth(a)` — среднее в окне 7×7 пикселей (приведение рельефа к разрешению поля);
2. `bg0`, `bg1` — аномалии Буге по исходным и уточнённым высотам: $\delta g - 0{,}1119 \cdot \overline{H}$;
3. `anomalies(bg, sigma_km, k)` — остаток после вычитания гауссова фона и маска структур $|\text{остаток}| > k\sigma$.

Выведите медианный сдвиг аномалии Буге в лесу, стандартное отклонение остатка, доли аномальной площади по исходным и уточнённым высотам и доли исчезнувших и появившихся структур.

In [ ]:
K_BOUGUER = 0.1119                                                       # Притяжение слоя 1 м плотностью 2670 кг/м³, мГал.

def smooth(a):
    """Среднее в окне 7×7 пикселей (210 м): осреднение до разрешения GGMplus."""
    return ...                                                           # uniform_filter(a, 7)

bg0 = ...                                                                # Аномалия Буге по исходным высотам: dg − K_BOUGUER · smooth(dem).
bg1 = ...                                                                # То же по уточнённым высотам dtm.

def anomalies(bg, sigma_km=3, k=1.5):
    """Остаток после вычитания регионального фона (гауссов фильтр) и маска аномальных структур |остаток| > k·σ."""
    resid = ...                                                          # bg − gaussian_filter(bg, sigma_km * 1000 / GRID["res"])
    return resid, ...                                                    # маска: |resid| > k · np.std(resid)

r0, a0 = anomalies(bg0)                                                  # По исходным высотам.
r1, a1 = anomalies(bg1)                                                  # По уточнённым высотам.
forest_s = uniform_filter((cls == 1).astype(float), 7) > 0.5             # Пиксели, где в окне 210 м преобладает лес.
# Ваш код: медианный сдвиг (bg1 − bg0) в лесу, σ остатка, доли аномальной площади, доли исчезнувших и появившихся структур

In [ ]:
fields = [(dg, "Возмущение силы тяжести δg", "RdBu_r"), (K_BOUGUER * smooth(dem), "Притяжение слоя 0,1119·H (исходные высоты)", "viridis"),
          (bg0, "Возмущение Буге δg − 0,1119·H", "RdBu_r"), (gaussian_filter(bg0, 3000 / GRID["res"]), "Региональный фон (σ = 3 км)", "RdBu_r"), (r0, "Остаток: локальные аномалии", "RdBu_r")]
fig, ax = plt.subplots(1, 5, figsize=(20, 4.2))                          # Пять карт в ряд.
for a, (arr, t, cmap) in zip(ax, fields):
    im = a.imshow(arr, cmap=cmap); a.set_title(t, fontsize=9); a.axis("off"); plt.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.show()                # Показываем рисунок.

**Вопрос:** Какую часть поля убирает каждый шаг цепочки: как выглядят возмущение, притяжение слоя, аномалия Буге, региональный фон и остаток? Какова амплитуда локальных аномалий на вашей территории?

**Ответ:** _______________

In [ ]:
fields_3d = [(dg, "Возмущение силы тяжести δg (измерено)", "RdBu_r"),
             (K_BOUGUER * smooth(dem), "Притяжение рельефа 0,1119·H (рассчитано по ЦМР)", "viridis"),
             (bg0, "Возмущение Буге δg − 0,1119·H", "RdBu_r"),
             (r0, "Остаток после вычитания фона: локальные аномалии", "RdBu_r")]
fig = plt.figure(figsize=(18, 12))                                       # Четыре поверхности: два ряда по две.
for i, (Z, t, cmap) in enumerate(fields_3d):
    ax = fig.add_subplot(2, 2, i + 1, projection="3d")
    surface_3d(ax, Z, cmap=cmap, z_ratio=0.3, title=t, zlabel="мГал")    # Высота поверхности – значение поля, мГал.
plt.tight_layout(); plt.show()

**Вопрос:** Что показывают трёхмерные поверхности полей? Повторяет ли поверхность возмущения рельеф вашей территории и что остаётся от этого сходства после вычитания притяжения слоя и фона?

**Ответ:** _______________

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 5))                            # Три рисунка рядом.
vmax = np.percentile(np.abs(r0), 98)                                     # Общая шкала для двух карт остатка.
for a, arr, msk, t in [(ax[0], r0, a0, "Исходные высоты"), (ax[1], r1, a1, "Уточнённые высоты")]:  # Две карты остатка:
    im = a.imshow(arr, cmap="RdBu_r", vmin=-vmax, vmax=vmax); plt.colorbar(im, ax=a, fraction=0.046, label="мГал")  # Остаток и шкала.
    a.contour(msk, levels=[0.5], colors="k", linewidths=0.6); a.set_title(f"{t}: остаточные аномалии и контуры структур")   # Границы структур.
im = ax[2].imshow(bg1 - bg0, cmap="viridis"); ax[2].set_title("Сдвиг аномалии Буге после уточнения"); plt.colorbar(im, ax=ax[2], fraction=0.046, label="мГал")  # Сдвиг Буге.
for a in ax: a.axis("off")                                               # Без осей координат.
plt.tight_layout(); plt.show()                   # Показываем рисунок.

**Вопрос:** Объясните своими словами, что показывает аномалия Буге и почему для неё важна точность модели рельефа. На сколько миллигал уточнение рельефа сдвинуло аномалию в лесу и как это соотносится со стандартным отклонением локальных аномалий? Совпадают ли изменившиеся структуры с границами леса?

**Ответ:** _______________

### **9.2. Чувствительность к параметрам фильтра и порога**

**Задание 9.2.** Задайте свои списки радиусов фона `SIGMAS` (км) и порогов `KS` (в σ) — не меньше трёх значений каждого — и сравните результаты.

In [ ]:
SIGMAS = [...]                                                           # Ваш код: радиусы регионального фона, км.
KS = [...]                                                               # Ваш код: пороги выделения структур, σ.
rows = []                                                                # Строки таблицы.
for sig_km in SIGMAS:                                                    # Радиус регионального фона, км.
    for kk in KS:                                                # Порог выделения структур, σ.
        r0_, m0_ = anomalies(bg0, sig_km, kk)                            # Структуры по исходным высотам.
        r1_, m1_ = anomalies(bg1, sig_km, kk)                            # Структуры по уточнённым высотам.
        rows.append([sig_km, kk, 100 * m1_.mean(), 100 * (m0_ & ~m1_).sum() / m0_.sum(), 100 * (m1_ & ~m0_).sum() / m1_.sum(), np.median((bg1 - bg0)[forest_s])])
sens = pd.DataFrame(rows, columns=["Фон, км", "Порог, σ", "Аномальная площадь, %", "Исчезло после уточнения, %", "Появилось, %", "Сдвиг Буге в лесу, мГал"]).round(2)
sens                                                                     # Показываем таблицу.

**Вопрос:** Как от радиуса фона и порога зависят доля аномальной площади и доля изменившихся после уточнения структур? Почему сдвиг аномалии Буге в лесу от этих параметров не зависит?

**Ответ:** _______________

### **9.3. Результаты на подложке Esri**

In [ ]:
fmap = folium.Map(location=center, zoom_start=11, tiles="Esri.WorldImagery")   # Подложка – космические снимки Esri.
cls_cmap = ListedColormap(["#00000000"] + [CLASS_COLORS[c] for c in range(1, 8)])   # Номер класса – номер цвета (0 – прозрачный).
overlay_layer(cls, "Карта покрова (модель ИИ)", cls_cmap, 0, 7, categorical=True).add_to(fmap)
overlay_layer(corr, "Поправка высот, м", "RdYlGn_r", 0, float(np.percentile(corr, 99))).add_to(fmap)
v = float(np.percentile(np.abs(r1), 98))                                 # Симметричная шкала аномалий.
overlay_layer(r1, "Локальные аномалии Буге (уточнённые высоты), мГал", "RdBu_r", -v, v, show=True).add_to(fmap)
folium.Rectangle([[ROI_BOUNDS[1], ROI_BOUNDS[0]], [ROI_BOUNDS[3], ROI_BOUNDS[2]]], color="yellow", weight=2, fill=False).add_to(fmap)
folium.LayerControl(collapsed=False).add_to(fmap)                        # Переключатель слоёв.
fmap                                                                     # Показываем карту.

**Вопрос:** Сопоставьте слои карты со снимком Esri: где находятся крупнейшие положительные и отрицательные локальные аномалии, связаны ли они с рельефом, лесом, водой или застройкой?

**Ответ:** _______________

---

## **10. Прикладная задача вашей зоны**

> 📚 **Подсказка:** лекция «Цифровая обработка спутниковых изображений рельефа и гравитационного поля», **разделы 8.7 и 9**.

Выполните подраздел, соответствующий прикладной задаче вашей зоны (столбец «Прикладная задача» в таблице зон); остальные подразделы — по желанию. Во всех задачах расчёт выполняется по исходной и по уточнённой модели рельефа.

### **10.1. Оценка зон затопления**

In [ ]:
water_lab, _ = label(cls == 6)                                           # Номера отдельных водных объектов (0 – суша).
if WATER_POINT is None:                                                  # Точка не задана:
    sizes = np.bincount(water_lab.ravel())                               # Число ячеек в каждом водном объекте.
    sizes[0] = 0                                                         # Сушу не учитываем.
    body = water_lab == np.argmax(sizes)                                 # Самый крупный водный объект.
else:                                                                    # Иначе:
    wr, wc = np.round(lonlat_to_rowcol(*WATER_POINT, GRID)).astype(int)  # Ячейка заданной точки.
    body = water_lab == water_lab[wr, wc]                                # Водный объект, содержащий точку.

def flood_area(height, rise):
    """Зона затопления при подъёме уровня водного объекта body на rise метров."""
    level = np.median(height[body]) + rise                               # Новый уровень воды, м.
    lab, _ = label(height <= level)                                      # Связные области ниже уровня.
    connected = np.unique(lab[body & (lab > 0)])                         # Области, которые касаются водного объекта.
    return np.isin(lab, connected) & (cls != 6), level                   # Затопленная суша и уровень.

f0, lev0 = flood_area(dem, WATER_RISE_M)                                 # По исходной модели.
f1, lev1 = flood_area(dtm, WATER_RISE_M)                                 # По уточнённой модели.
b1 = pd.DataFrame([["Исходная модель", lev0, f0.sum() * 900 / 1e6], ["Уточнённая модель", lev1, f1.sum() * 900 / 1e6]],  # Уровень и площадь, км².
                  columns=["Модель", "Уровень воды, м", "Площадь затопления, км²"]).round(2)
if max(f0.mean(), f1.mean()) > 0.25:                                     # Затоплено больше четверти участка:
    print("Затоплено больше четверти территории. Для широкой поймы это возможно; если же выбран водоём выше "
          "окружающей местности (пруд, хвостохранилище), укажите точку на реке в параметре WATER_POINT.")
flood_colors = ListedColormap([(0, 0, 0, 0), "#2e86de", "#e74c3c"])      # Прозрачный, вода, затопление.
fig, ax = plt.subplots(1, 2, figsize=(13, 6))                            # Два рисунка рядом.
for a, f, t in [(ax[0], f0, "Исходная модель"), (ax[1], f1, "Уточнённая модель")]:  # Две карты затопления:
    layer = np.where(cls == 6, 1, np.where(f, 2, 0))                     # 1 – вода, 2 – затопленная суша.
    a.imshow(ls.hillshade(dtm, vert_exag=3, dx=30, dy=30), cmap="gray")  # Подложка – отмывка рельефа.
    a.imshow(layer, cmap=flood_colors, interpolation="nearest")          # Вода и зона затопления.
    a.set_title(f"{t}: зона затопления +{WATER_RISE_M:g} м (красный)"); a.axis("off")  # Заголовок, без осей.
plt.tight_layout(); plt.show()                  # Показываем рисунок.
b1                                                                       # Показываем таблицу.

### **10.2. Профиль линейного объекта**

In [ ]:
if PROFILE is None:                                                      # По умолчанию – диагональ участка.
    PROFILE = [(ROI_BOUNDS[0] + 0.01, ROI_BOUNDS[1] + 0.01), (ROI_BOUNDS[2] - 0.01, ROI_BOUNDS[3] - 0.01)]
n_pts = 400                                                              # Число точек профиля.
lons_p = np.linspace(PROFILE[0][0], PROFILE[1][0], n_pts)                # Долготы точек профиля.
lats_p = np.linspace(PROFILE[0][1], PROFILE[1][1], n_pts)                # Широты точек профиля.
pr, pc = lonlat_to_rowcol(lons_p, lats_p, GRID)                          # Положение точек на сетке.
h0 = map_coordinates(dem, [pr, pc], order=1)                             # Высоты по исходной модели.
h1 = map_coordinates(dtm, [pr, pc], order=1)                             # Высоты по уточнённой модели.
Xp, Yp = Transformer.from_crs("EPSG:4326", GRID["crs"], always_xy=True).transform(lons_p, lats_p)   # Точки в метрах.
sp = np.r_[0, np.cumsum(np.hypot(np.diff(Xp), np.diff(Yp)))] / 1000      # Расстояние вдоль профиля, км.
plt.figure(figsize=(13, 4))                                              # Широкий рисунок для профиля.
plt.plot(sp, h0, color="grey", label="Исходная модель"); plt.plot(sp, h1, color="#d35400", label="Уточнённая модель")  # Два профиля.
plt.xlabel("км"); plt.ylabel("м"); plt.legend(); plt.title("Продольный профиль"); plt.show()                                           # Подписи и показ рисунка.

def profile_slope(h):
    """Уклон между соседними точками профиля, градусы."""
    return np.degrees(np.arctan(np.abs(np.diff(h)) / np.diff(sp * 1000)))

slope0, slope1 = profile_slope(h0), profile_slope(h1)                    # Уклоны по двум моделям.
b2 = pd.DataFrame([["Исходная модель", h0.max() - h0.min(), np.percentile(slope0, 95)],  # Перепад и уклон по моделям.
                   ["Уточнённая модель", h1.max() - h1.min(), np.percentile(slope1, 95)]],
                  columns=["Модель", "Перепад высот, м", "Уклон, 95-й процентиль, °"]).round(2)
b2                                                                       # Показываем таблицу.

### **10.3. Объёмы земляных масс**

In [ ]:
if VOLUME_BOX is None:                                                   # Участок не задан – выбираем автоматически:
    k = int(2000 / GRID["res"])                                          # Сторона квадрата 2 км в ячейках.
    mean_corr = uniform_filter(corr, k)                                  # Средняя поправка в окне 2×2 км вокруг каждой ячейки.
    inner = mean_corr[k // 2:-k // 2, k // 2:-k // 2]                    # Без краёв, где окно выходит за участок.
    r_, c_ = np.unravel_index(np.argmax(inner), inner.shape)             # Окно с наибольшей средней поправкой.
    box = (slice(r_, r_ + k), slice(c_, c_ + k))                         # Строки и столбцы квадрата.
else:                                                                    # Иначе:
    r_a, c_a = np.round(lonlat_to_rowcol(VOLUME_BOX[0], VOLUME_BOX[3], GRID)).astype(int)   # Северо-западный угол.
    r_b, c_b = np.round(lonlat_to_rowcol(VOLUME_BOX[2], VOLUME_BOX[1], GRID)).astype(int)   # Юго-восточный угол.
    box = (slice(r_a, r_b), slice(c_a, c_b))                             # Строки и столбцы участка.
if VOLUME_LEVEL is None:                                                 # Объём над самой низкой точкой.
    base = dtm[box].min()                                                # Базовая отметка – минимум уточнённой модели.
    v0 = (dem[box] - base).clip(0).sum() * 900                           # Объём по исходной модели, м³.
    v1 = (dtm[box] - base).clip(0).sum() * 900                           # Объём по уточнённой модели, м³.
else:                                                                    # Объём выемки ниже отметки.
    v0 = (VOLUME_LEVEL - dem[box]).clip(0).sum() * 900                   # Выемка по исходной модели, м³.
    v1 = (VOLUME_LEVEL - dtm[box]).clip(0).sum() * 900                   # Выемка по уточнённой модели, м³.
b3 = pd.DataFrame([["Исходная модель", v0 / 1e6], ["Уточнённая модель", v1 / 1e6], ["Разница, %", 100 * (v0 - v1) / v1]],  # Объёмы, млн м³.
                  columns=["", "Объём, млн м³"]).round(2)
fig, ax = plt.subplots(1, 3, figsize=(16, 5))                            # Три рисунка рядом.
for a, h, t in [(ax[0], dem, "Исходная модель"), (ax[1], dtm, "Уточнённая модель")]:  # Отмывка участка по двум моделям:
    a.imshow(ls.hillshade(h[box], vert_exag=3, dx=30, dy=30), cmap="gray"); a.set_title(t); a.axis("off")
im = ax[2].imshow((dem - dtm)[box], cmap="RdYlGn_r"); ax[2].set_title("Разница высот, м"); ax[2].axis("off")  # Разница моделей.
plt.colorbar(im, ax=ax[2], fraction=0.046)
plt.tight_layout(); plt.show()                  # Показываем рисунок.
b3                                                                       # Показываем таблицу.

### **10.4. Углублённый гравиметрический анализ: типы аномальных структур**

In [ ]:
amp = r1.ravel()                                                         # Амплитуда остаточной аномалии, мГал.
slope_s = smooth(REL["slope"]).ravel()                                   # Средний уклон в окне 210 м.
forest_share = uniform_filter((cls == 1).astype(float), 7).ravel()       # Доля леса в окне.
height_s = smooth(dtm).ravel()                                           # Средняя высота в окне.
feat_g = np.column_stack([amp, slope_s, forest_share, height_s])         # Признаки каждой ячейки.
sel = a1.ravel()                                                         # Только ячейки аномальных структур.
X_g = StandardScaler().fit_transform(feat_g[sel])                        # Стандартизованные признаки аномальных ячеек.
sub = np.random.default_rng(SEED).choice(len(X_g), min(20000, len(X_g)), replace=False)   # Выборка ячеек для силуэта.
K_RANGE = list(range(2, 9))                                              # Проверяемое число типов: от 2 до 8.
rows = []                                                                # Строки будущей таблицы.
for k_ in K_RANGE:                                                       # Для каждого числа типов:
    km_k = KMeans(k_, n_init=10, random_state=SEED).fit(X_g)             # Разбиение на k_ типов.
    rows.append([k_, km_k.inertia_, silhouette_score(X_g[sub], km_k.labels_[sub])])   # Сумма квадратов и силуэт.
k_sel = pd.DataFrame(rows, columns=["Число типов", "Внутрикластерная сумма квадратов", "Силуэт"])   # Таблица выбора.
wcss = k_sel["Внутрикластерная сумма квадратов"]                         # Короткое имя для расчёта.
x_n = (k_sel["Число типов"] - K_RANGE[0]) / (K_RANGE[-1] - K_RANGE[0])   # Число типов, приведённое к диапазону 0…1.
y_n = (wcss - wcss.min()) / (wcss.max() - wcss.min())                    # Сумма квадратов, приведённая к 0…1.
k_sel["Расстояние до хорды"] = (1 - x_n) - y_n                           # Насколько точка ниже прямой между крайними точками.
K_TYPES = int(k_sel.loc[k_sel["Расстояние до хорды"].idxmax(), "Число типов"])   # Точка наибольшего излома – «локоть».
fig, ax = plt.subplots(1, 2, figsize=(12, 4))                            # Два графика рядом.
ax[0].plot(k_sel["Число типов"], wcss, "o-", label="сумма квадратов")    # Кривая локтя.
ax[0].plot([K_RANGE[0], K_RANGE[-1]], [wcss.iloc[0], wcss.iloc[-1]], "--", color="grey", label="хорда")   # Прямая между крайними точками.
ax[0].axvline(K_TYPES, color="#d35400", ls=":", label=f"выбрано: {K_TYPES}")   # Выбранное число типов.
ax[0].set_title("Метод локтя"); ax[0].set_xlabel("Число типов"); ax[0].set_ylabel("Внутрикластерная сумма квадратов"); ax[0].legend()
ax[1].plot(k_sel["Число типов"], k_sel["Силуэт"], "o-"); ax[1].axvline(K_TYPES, color="#d35400", ls=":")   # Силуэт при каждом k.
ax[1].set_title("Коэффициент силуэта"); ax[1].set_xlabel("Число типов"); ax[1].set_ylabel("Силуэт")
plt.tight_layout(); plt.show()                # Показываем рисунок.
k_sel = k_sel.round(3)                                                   # Округляем для показа и файла.
print(f"Число типов по методу локтя: {K_TYPES}")
k_sel                                                                    # Показываем таблицу.

In [ ]:
km = make_pipeline(StandardScaler(), KMeans(K_TYPES, n_init=10, random_state=SEED)).fit(feat_g[sel])   # Стандартизация и разбиение на выбранное число типов.
types = np.full(a1.size, -1)                                             # −1 – ячейка вне структур.
types[sel] = km.predict(feat_g[sel])                                     # Номер типа для аномальных ячеек.
types = types.reshape(a1.shape)                                          # Преобразуем обратно в карту.
rows = []                                                                # Строки будущей таблицы.
for t in range(K_TYPES):                                                 # Для каждого типа:
    m_ = types == t                                                      # Ячейки типа t.
    rows.append({"Тип": t + 1, "Доля аномальной площади, %": 100 * m_.sum() / sel.sum(),
                 "Средняя аномалия, мГал": r1[m_].mean(), "Уклон, °": REL["slope"][m_].mean(), "Доля леса": (cls[m_] == 1).mean()})
b4 = pd.DataFrame(rows).round(2)                                         # Описание типов.
type_colors = ["#1b6ca8", "#e67e22", "#7d3c98", "#27ae60", "#c0392b", "#16a085", "#8d6e63", "#7f8c8d"][:K_TYPES]   # Цвета типов.
fig, ax = plt.subplots(figsize=(8, 7))                                   # Один рисунок.
ax.imshow(ls.hillshade(dtm, vert_exag=3, dx=30, dy=30), cmap="gray")     # Подложка – отмывка рельефа.
ax.imshow(np.ma.masked_less(types, 0), cmap=ListedColormap(type_colors), vmin=-0.5, vmax=K_TYPES - 0.5, alpha=0.8, interpolation="nearest")   # Ячейки вне структур прозрачны.
ax.legend(handles=[Patch(color=type_colors[t], label=f"Тип {t + 1}") for t in range(K_TYPES)], loc="lower left")  # Легенда типов.
ax.set_title("Типы аномальных структур"); ax.axis("off")                 # Заголовок, без осей.
plt.show()                                        # Показываем рисунок.
b4                                                                       # Показываем таблицу.

In [ ]:
feat_g0 = np.column_stack([r0.ravel(), smooth(relief_features(dem)["slope"]).ravel(),   # Те же признаки,
                           uniform_filter((cls == 1).astype(float), 7).ravel(), smooth(dem).ravel()])   # но по исходной модели.
types0 = np.full(a0.size, -1)                                            # −1 – ячейка вне структур.
types0[a0.ravel()] = km.predict(feat_g0[a0.ravel()])                     # Типы по исходным высотам той же моделью.
types0 = types0.reshape(a0.shape)                                        # Обратно в карту.
both = a0 & a1                                                           # Аномалия есть в обоих вариантах.
rows = []                                                                # Строки будущей таблицы.
for t in range(K_TYPES):                                                 # Для каждого типа:
    share0 = 100 * (types0 == t).sum() / a0.sum()                        # доля по исходным высотам,
    share1 = 100 * (types == t).sum() / a1.sum()                         # доля по уточнённым высотам,
    switched = ((types0 == t) & (types != t) & both).sum()               # ячейки, сменившие этот тип на другой.
    rows.append([f"Тип {t + 1}", share0, share1, 100 * switched / max(((types0 == t) & both).sum(), 1)])
b4_change = pd.DataFrame(rows, columns=["Тип", "Доля площади, исходные высоты, %", "Доля площади, уточнённые высоты, %", "Сменили тип, %"]).round(1)
print(f"Аномальная площадь: {a0.sum() * 900 / 1e6:.1f} км² (исходные высоты) и {a1.sum() * 900 / 1e6:.1f} км² (уточнённые); "
      f"общая {both.sum() * 900 / 1e6:.1f} км²; сменили тип {100 * (both & (types0 != types)).sum() / both.sum():.1f}% общей площади")
b4_change                                                                # Показываем таблицу.

### **10.5. Высота растительности**

In [ ]:
veg = (dem - dtm).clip(0)                                                # Разность моделей; отрицательные значения → 0.
fm = cls == 1                                                            # Ячейки леса.
b5 = pd.DataFrame([["Медиана в лесу, м", np.median(veg[fm]), np.median(stack["eth"][fm])],  # Медианы и корреляция.
                   ["Корреляция с картой ETH (лес)", np.corrcoef(veg[fm], stack["eth"][fm])[0, 1], np.nan]],
                  columns=["Показатель", "Разность моделей", "Карта ETH"]).round(2)
fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))                          # Два рисунка рядом.
im = ax[0].imshow(np.where(fm, veg, np.nan), cmap="YlGn", vmin=0, vmax=np.nanpercentile(veg[fm], 98))   # Только лес.
ax[0].set_title("Высота растительности по разности моделей, м"); ax[0].axis("off")  # Заголовок, без осей.
plt.colorbar(im, ax=ax[0], fraction=0.046)                               # Шкала значений.
sub = np.random.default_rng(SEED).choice(np.where(fm.ravel())[0], min(20000, fm.sum()), replace=False)   # 20 000 ячеек леса для графика.
ax[1].scatter(stack["eth"].ravel()[sub], veg.ravel()[sub], s=1, alpha=0.3)  # Каждая точка – ячейка леса.
ax[1].set_xlabel("Карта ETH, м"); ax[1].set_ylabel("Разность моделей, м"); ax[1].set_title("Сравнение с картой высоты леса ETH")  # Подписи.
plt.tight_layout(); plt.show()            # Показываем рисунок.
b5                                                                       # Показываем таблицу.

**Вопрос по прикладной задаче своей зоны.** Приведите результат по исходной и по уточнённой модели и разницу между ними (в единицах задачи и в процентах). Объясните, откуда берётся разница, и укажите ограничения расчёта:
- *затопление* — уровень воды и площадь зоны затопления по двум моделям;
- *профиль* — перепад высот и крутизна уклонов по двум моделям, где на профиле уточнение изменило высоты;
- *объёмы* — объём по двум моделям и разница в процентах, есть ли на участке реальные изменения рельефа между съёмками;
- *гравиметрический анализ* — число типов и как оно выбрано, чем типы различаются, сменили ли структуры тип после уточнения;
- *высота растительности* — медианы разности моделей и карты ETH в лесу, корреляция, почему разность меньше высоты деревьев.

**Ответ:** _______________

---

## **11. Итоговые выводы**

В выводах обязательно отразить (с числами для вашей зоны):
1. **Изображения и фильтры**: какие операции над изображениями использованы на каждом этапе (композит, индексы, фильтры в окне, интерполяция, пороговая сегментация, связные области);
2. **Ошибка глобальной модели рельефа**: завышение в лесу и на открытой местности до и после уточнения, RMSE до и после;
3. **Классификация**: точность при случайной и пространственной проверке, вклад признаков рельефа;
4. **Аномалии Буге**: сдвиг в лесу, доля изменившихся структур, чувствительность к параметрам;
5. **Прикладная задача**: результат по двум моделям и его практическое значение.

**Выводы:** _______________

---

## **Что сдавать**

Выполненный блокнот, сохранённый вместе с выводами ячеек:
1. Код всех заданий и результаты его выполнения.
2. В выводах ячеек — все таблицы, графики, карты, трёхмерные поверхности, интерактивная трёхмерная модель и интерактивные карты на подложке Esri (перед сохранением выполните все ячейки по порядку и убедитесь, что выводы не очищены).
3. Ответы на все вопросы — в ячейках **Ответ:** под соответствующими результатами, и итоговые выводы в разделе 11.

## **Дополнительные задания (выполняются по желанию)**

1. **Другая прикладная задача.** Выполните ещё одну задачу раздела 10 и сравните, какая из них сильнее зависит от уточнения рельефа.
2. **Медианный фильтр в пространстве.** Сгладьте изображение рельефа медианным фильтром (`scipy.ndimage.median_filter`) и средним в том же окне; сравните изображения высоких частот и объясните различие на бровках и тальвегах.
3. **Фон не гауссовым фильтром.** Оцените региональный фон аномалий Буге средним в квадратном окне вместо гауссова фильтра и сравните доли аномальной площади.
4. **Другая модель рельефа.** Повторите уточнение и расчёт аномалий Буге для ALOS AW3D30 (`stack["alos"]`, высоты над EGM96) и сравните результаты с Copernicus.